# EMBER end to end, background-whitened + target-sharpened: §3.1 Sparse MF → §3.2 concept features → §3.3 embedding erasure

> **Background-whitened + target-sharpened variant** of `ember_three_steps.ipynb` (method: `Instructions.txt`, code:
> `experiments/whitening/sharpen.py`). Same concept, data, hyperparameters and seed. Step 3.1 factorizes
> $X = (E_{V^*} - \mathbf 1\boldsymbol\mu_B^\top)\,M$ with $M = W\,S_\eta^{\gamma/2}$ instead of the raw rows:
>
> - $W = Q(\Lambda+\epsilon I)^{-1/2}Q^\top$ whitens against a **background** that is fixed per model (`BACKGROUND`:
>   `"wiki"` = token types of a large Wikipedia sample, `"full"` = whole vocabulary), not against $V^*$ itself.
> - $S_\eta = (1-\eta)\,\tfrac1m U_C^\top U_C + \eta I$ is the second moment of the whitened **target** rows (`TARGET`:
>   all tokens of the concept's English Wikipedia page, or those minus the Wikipedia background types). Its eigenvalues
>   are enrichment ratios; $S_\eta^{\gamma/2}$ stretches enriched directions (`GAMMA` = 0: whitening only).
> - **3.1.c′** builds the transform `T` (`T.forward`, `T.unmap`, `T.matrix`). **3.1.d** fits `Z`, `Y` on $X$;
>   **3.1.d′** fits `Z0`, `Y0` on the raw matrix (cached once per notebook in `ORIG_DIR`). 3.1.e/f report both.
> - **3.2** selects $F_C$ from the transformed fit; **3.2.e′** runs the same selection on the original fit (`F_C0`);
>   **3.2.w** matches the two fits' features by support overlap and writes `RUN_DIR/summary.json`.
> - **3.3** maps the concept part back exactly, $\mathbf c_t = T^{-1}\tilde{\mathbf c}_t$ (the mean cancels), and edits the raw embeddings.
>
> **Runs.** Every expensive result is cached in `RUN_DIR = runs/whiten/<model>/<concept>/<BACKGROUND>_<TARGET>_g<GAMMA>/`
> (`smf.pt`, `judge.json`, `mc.json`, `gen.json`, `summary.json`); the original fit and its judge in `ORIG_DIR`.
> `SMOKE = True` makes a quick end-to-end pass into `..._smoke`. `scripts/run_overnight.py --queue runs/whiten/queue.json`
> runs the sweep headless (`--param BACKGROUND=... GAMMA=...`), stops before Step 3.3 for the sweep entries, and skips the
> §3.4 cells (tagged `skip-overnight`).


Paper: *Don't Forget Your Embeddings: Robust Knowledge Erasure via Precise Editing of Embeddings*
(Suslik, Shafran, Geva, EMNLP Findings 2026). This notebook runs the three stages of §3 for one concept,
using the paper's default hyperparameters. Each stage is split into small blocks, and each block starts
with a note on what it does.

| Step | Paper section | What happens | Repo code reused |
|---|---|---|---|
| **3.1** | Finding Embedding Features with Sparse MF | factorize $E_{V^*}^\top \approx ZY$ with WTA-sparse $Y$ | `SparseMatrixFactorization`, `fit_with_ridge` |
| **3.2** | Identifying Concept-Related Features | mass ratio $\rho_i \ge \tau$, then a two-stage Gemini judge → $F_C$ | `compute_embedding_stats`, prompts from `ember/interpret_features.py` |
| **3.3** | Erasing Concept-Related Features | $\mathbf e_t \leftarrow \mathbf e_t - \delta \sum_{i\in F_C} Y_{i,t}\,\mathbf z_i$ for concept tokens | same math as `ember/erasure/embed_edit.py` |

**Notation.** Paper $Z \in \mathbb R^{d\times k}$ = code `F_`; paper $Y \in \mathbb R^{k\times|V^*|}$ = code `G_.T`.
In this notebook, `Z` and `Y` use the paper's shapes.

**Requirements:** a GPU, `HF_TOKEN` (only needed for gated models such as Llama), and a Gemini API key for Step 3.2.

---
## 0. Setup

### 0.1 Hyperparameters (paper defaults)

| Symbol | Value | Source |
|---|---|---|
| $k$ (rank) | 100 for Gemma-2-2B-it, 200 for Llama-3.1-8B-Instruct | App. A.1 |
| $s$ (WTA sparsity on $Y$) | 0.01, so each factor keeps $\lceil 0.01\,|V^*| \rceil$ tokens | App. A.1 |
| $\lambda$ (ridge) | $10^{-4}$ | App. A.1 |
| $T$ / patience / tol | 20,000 iters / 500 iters / $10^{-4}$ | App. A.1 |
| $\|S_C\| = \|S_N\|$ | 300 sentences each (`data/*_sentences.json`) | §4.1 |
| $\tau$ (mass-ratio threshold) | 2.0 | App. A.3 |
| judge confidence | ≥ 0.85, with 20 tokens per feature | App. A.3 |
| $\delta$ (edit strength) | tuned per concept over {0.5, 1, 2, 5, 10, 50, 100, 200} | App. C.3 |

In [ ]:
import gc
import json
import math
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from dotenv import load_dotenv
from tqdm.auto import tqdm

load_dotenv()  # HF_TOKEN / GEMINI_API_KEY from .env, if present
pd.set_option("display.max_colwidth", 200)

# ---- what to erase ------------------------------------------------------
CONCEPT = "Harry Potter"
MODEL_NAME = "google/gemma-2-2b-it"     # or "meta-llama/Llama-3.1-8B-Instruct"
SEED = 42

# ---- Step 3.1: Sparse MF (App. A.1) -------------------------------------
RANK = 200 if "llama" in MODEL_NAME.lower() else 100   # k
SPARSITY = 0.01          # s
RIDGE = 1e-4             # lambda
MAX_ITER = 20_000        # T
PATIENCE = 200

# ---- Step 3.2: feature selection (App. A.3) -----------------------------
TAU = 2.0
CONFIDENCE = 0.85
TOP_K_TOKENS = 20
GEMINI_MODEL = "models/gemini-3.5-flash-lite"   # repo default; the paper used Gemini-2.5-Flash-Lite
JUDGE_FORMAT = "tokens"   # Stage-1 judge input: "tokens" (bare tokens, paper; this notebook's original format)
                          # | "words" (token → the words it was cut from in S_C ∪ S_N, alphabetical)

# ---- Step 3.3: erasure (App. C.3) ---------------------------------------
DELTA_GRID = [0.5, 1.0, 2.0, 5.0, 10.0, 50.0, 100.0, 200.0]
DELTA = 10.0             # repo demo's pick for Harry Potter / Gemma; use the sweep in 3.3.f for other concepts

# ---- background whitening + target sharpening (Instructions.txt, experiments/whitening/sharpen.py) ----
import re
import sys

sys.path.insert(0, "experiments/whitening")
from sharpen import BackgroundWhitener, Sharpener, Transform, principal_angle_sines

BACKGROUND = "wiki"      # "wiki" (Wikipedia token types) or "full" (whole vocabulary)
TARGET = "page"          # "page" (all tokens of the concept's English Wikipedia page) or "page_minus_wiki"
GAMMA = 1.0              # sharpening strength; 0 = background whitening only
MODEL_SLUG = "gemma"     # "gemma" | "llama": selects experiments/whitening/data/<MODEL_SLUG>
SMOKE = False            # True: quick end-to-end pass (short fit, 3 judged features, tiny evals)

# ---- overrides from scripts/run_overnight.py --param NAME=VALUE (set in an injected first cell) ----
globals().update(globals().get("_OVERRIDES", {}))
assert BACKGROUND in ("wiki", "full"), BACKGROUND
assert TARGET in ("page", "page_minus_wiki"), TARGET
assert GAMMA >= 0, GAMMA
assert MODEL_SLUG == ("llama" if "llama" in MODEL_NAME.lower() else "gemma"), (MODEL_SLUG, MODEL_NAME)

WH_DIR = Path("experiments/whitening/data") / MODEL_SLUG
_defaults_path = WH_DIR / "defaults.json"
if _defaults_path.exists():
    defaults = json.loads(_defaults_path.read_text())
    print(f"whitening defaults from {_defaults_path}: {defaults}")
else:
    defaults = {"eps_rel": 1e-3, "eta": 0.2}
    print(f"{_defaults_path} not found: built-in defaults {defaults}")
ETA = globals().get("_OVERRIDES", {}).get("ETA", defaults["eta"])              # target shrinkage (fixed hyperparameter)
EPS_REL = globals().get("_OVERRIDES", {}).get("EPS_REL", defaults["eps_rel"])  # eps = EPS_REL * mean(background eigenvalues)

if SMOKE:
    MAX_ITER, PATIENCE = 50, 5

# ---- run directories: resume caches ----
concept_slug = re.sub(r"\W+", "_", CONCEPT).strip("_")
_smoke = "_smoke" if SMOKE else ""
RUN_DIR = Path("runs/whiten") / MODEL_SLUG / concept_slug / f"{BACKGROUND}_{TARGET}_g{GAMMA:g}{_smoke}"
ORIG_DIR = Path("runs/whiten") / MODEL_SLUG / concept_slug / f"original{_smoke}"   # original fit + judge, shared by all configs
RUN_DIR.mkdir(parents=True, exist_ok=True)
ORIG_DIR.mkdir(parents=True, exist_ok=True)


def load_json(path: Path, default):
    return json.loads(path.read_text()) if path.exists() else default


def save_json(path: Path, obj) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")   # write then rename, so a crash never leaves a half file
    tmp.write_text(json.dumps(obj, indent=1, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
    tmp.replace(path)


def invalidate(directory: Path, names) -> None:
    """Move stale cache files aside (<name>.stale) so they are recomputed."""
    for name in names:
        p = directory / name
        if p.exists():
            p.replace(p.with_name(p.name + ".stale"))
            print(f"  moved stale {p} -> {p.name}.stale")

concepts = [e["concept"] for e in json.loads(Path("data/concept_sentences.json").read_text())]
assert CONCEPT in concepts, f"{CONCEPT!r} is not one of: {concepts}"
print(f"concept={CONCEPT!r} | model={MODEL_NAME} | k={RANK}")
print(f"background={BACKGROUND} | target={TARGET} | gamma={GAMMA:g} | eta={ETA:g} | eps_rel={EPS_REL:g} | "
      f"smoke={SMOKE} | run dir={RUN_DIR} | original-fit dir={ORIG_DIR}")

### 0.2 Gemini API key

Step 3.2 is the only step that calls Gemini. You can paste your key below or put `GEMINI_API_KEY=...`
in `.env`. Clear the key from this cell before you commit or share the notebook.

In [ ]:
GEMINI_API_KEY = ""  # paste here, or leave empty to use .env / the environment

if GEMINI_API_KEY:
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
assert any(os.getenv(k) for k in ("GOOGLE_API_KEY", "GEMINI_API_TOKEN", "GEMINI_API_KEY")), \
    "No Gemini key found: Step 3.2 needs one."
print("Gemini key found.")

Gemini key found.


### 0.3 Repo imports

These are the building blocks the paper's pipeline uses. Reusing them means the notebook gives the same
results as `ember.train_mf_features`, `ember.interpret_features`, and `ember.run_erasure`.

In [ ]:
from llm_utils.activation_generator import ActivationGenerator, extract_token_ids_sample_ids_and_labels
from transformers import AutoTokenizer

from ember.erasure.model_loader import load_hf_model
from ember.evals.mc import evaluate_mc_generation, prepare_mc_items
from ember.evals.model_wrap import WrappedHFModel
from ember.interpret_features import (
    MEMBERSHIP_PROMPT,
    TOKENS_DESC_SYS,
    SimpleGeminiClient,
    classify_membership_with_desc,
    get_description,
)
from ember.local_datasets import ConceptDataset, load_mc_qa_items, load_open_qa_examples
from ember.timing import Timer
from ember.utils import (
    SparseMatrixFactorization,
    _safe_model_name,
    build_token_label_codes,
    collect_feature_rows_for_embeddings,
    compute_embedding_stats,
    fit_with_ridge,
    get_embedding_matrix,
    get_special_token_ids,
    resolve_device,
    set_seed,
)

set_seed(SEED)
# DEVICE = resolve_device("auto")
# This host's vGPU reports CUDA as available but prohibits compute, so force CPU.
DEVICE = "cpu"
os.environ["EMBER_DEVICE"] = "cpu"  # load_hf_model reads this
print(f"device: {DEVICE}")

/homes/orip/.conda/envs/Ember/lib/python3.10/site-packages/google/api_core/_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.10.21). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.api_core.
  warnings.warn(message, FutureWarning)


device: cpu


---
# Step 3.1: Finding embedding features with Sparse Matrix Factorization

**Goal:** split each token embedding $\mathbf e_t$ into a sum of a few shared directions, so that
semantically related tokens (e.g. *Harry*, *Hogwarts*, *wand*) share a feature:

$$E_{V^*}^\top \approx Z\,Y,\qquad Z\in\mathbb R^{d\times k}\ \text{(dense features)},\quad Y\in\mathbb R^{k\times|V^*|}\ \text{(WTA-sparse coefficients)}$$

The hard winner-takes-all (WTA) constraint on the rows of $Y$ means each feature $\mathbf z_i$ is used by only
1% of the tokens in $V^*$. That is why a feature can be specific to one concept.

### 3.1.a Load the model (TransformerLens)

The pipeline reads $E$ through TransformerLens. **Note:** for Gemma, TL folds the $\sqrt{d}$ embedding
multiplier into `W_E`, so the factorization runs in this *scaled* space. Step 3.3 accounts for that
when it writes back into the HF weights.

In [ ]:
act_generator = ActivationGenerator(MODEL_NAME, model_device=DEVICE, data_device="cpu", mode="mlp")
tl_model = act_generator.model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

E = get_embedding_matrix(tl_model)  # (|V|, d)
VOCAB_SIZE, D_MODEL = E.shape
print(f"E: {tuple(E.shape)}  dtype={E.dtype}")

Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.92s/it]


Loaded pretrained model google/gemma-2-2b-it into HookedTransformer
E: (256000, 2304)  dtype=torch.float32


### 3.1.b Build $S_C \cup S_N$ and the token set $V^*$

- $S_C$: 300 Wikipedia sentences about the concept. $S_N$: 300 neutral Wikipedia sentences.
- $V^*$: the unique non-special token ids in $S_C \cup S_N$. The model factorizes only these rows of $E$,
  which is cheaper and keeps the factorization focused on the concept. The neutral tokens let Step 3.2
  tell concept features apart from general ones.

In [ ]:
dataset = ConceptDataset(concept_name=CONCEPT, neutral_sample_seed=42)
token_ids_all, sample_ids, labels_all = extract_token_ids_sample_ids_and_labels(dataset, act_generator)

special = get_special_token_ids(tl_model)
vstar_ids = sorted({int(t) for t in token_ids_all.tolist()} - special)

n_concept_sents = sum(lbl == CONCEPT for _, lbl in dataset)
print(f"|S_C| = {n_concept_sents}   |S_N| = {len(dataset) - n_concept_sents}")
print(f"token occurrences = {len(token_ids_all)}  ->  |V*| = {len(vstar_ids)} "
      f"({len(vstar_ids) / VOCAB_SIZE:.2%} of the vocabulary)")

Extracting token IDs: 100%|██████████| 120/120 [00:00<00:00, 1111.53it/s]

|S_C| = 300   |S_N| = 300
token occurrences = 18611  ->  |V*| = 4780 (1.87% of the vocabulary)


### 3.1.c Build the data matrix $A = E_{V^*}^\top$

Each column of $A$ is one token's embedding.

In [ ]:
E_vstar = E.index_select(0, torch.tensor(vstar_ids, device=E.device)).detach().float()  # (|V*|, d)
A = E_vstar.T.contiguous().to(DEVICE)                                                      # (d, |V*|)

print(f"A: {tuple(A.shape)}  (d x |V*|)")
print(f"WTA keeps ceil(s*|V*|) = {math.ceil(SPARSITY * len(vstar_ids))} tokens per feature")

A: (2304, 4780)  (d x |V*|)
WTA keeps ceil(s*|V*|) = 48 tokens per feature


### 3.1.c′ Background whitening and target sharpening

Instructions steps 1–4, in the row convention of the notebook (rows are tokens; `sharpen.py`):

$$U = (E_{V^*} - \mathbf 1\boldsymbol\mu_B^\top)\,W,\quad W = Q(\Lambda+\epsilon I)^{-1/2}Q^\top;\qquad
S_\eta = (1-\eta)\,\tfrac1m U_C^\top U_C + \eta I = P\,\mathrm{diag}(g)\,P^\top;\qquad X = U\,S_\eta^{\gamma/2}$$

1. **Background** ($\boldsymbol\mu_B$, $Q$, $\Lambda$ of $\Sigma_B$) is loaded from `WH_DIR/bg_<BACKGROUND>_stats.pt`. It was computed once per
   model, in the same space as `E` (TL `W_E`: Gemma rows already × $\sqrt d$; Llama unscaled). $\epsilon$ = `EPS_REL` × mean($\Lambda$).
   Check: $W\Sigma_B W = Q\,\mathrm{diag}\big(\lambda/(\lambda+\epsilon)\big)Q^\top \approx I$, except along the smallest
   eigenvalues, which $\epsilon$ deliberately damps.
2. **Whitening** centres on $\boldsymbol\mu_B$ (not on the mean of $V^*$), so the concept's shift away from the background stays in the data.
3. **Target**: the token ids of the concept's Wikipedia page (`WH_DIR/targets/<concept>.json`, key `TARGET`), whitened with the
   same $W$. The second moment is taken around 0, so it includes that shift. $g_j$ are **enrichment ratios** ($g_j > 1$: the target
   varies more than the background along $\mathbf p_j$); $\eta$ = `ETA` puts a floor under the directions the target does not use.
4. **Sharpening** with $S_\eta^{\gamma/2}$ applies one fixed map to every row of $V^*$, concept and neutral alike. `GAMMA = 0` skips it.

The combined map is `T` (`Transform`): `T.forward(rows)` $= (\text{rows}-\boldsymbol\mu_B)\,M$ with $M$ = `T.matrix` $= W S_\eta^{\gamma/2}$, and
`T.unmap(z)` $= z\,S_\eta^{-\gamma/2}W^{-1}$ maps directions back. `transform(rows)` is the float32 helper the analysis cells use.

**Diagnostics.** The largest enrichment ratios, and the contrastive-PCA directions $\mathbf p_1,\dots,\mathbf p_5$: the whitened
Wikipedia background token types (`bg_wiki_types.json`) projected on each, top 15 shown (`*` = in the target set). If they are
concept tokens, the signal is there before any factorization. With `GAMMA = 0` the same diagnostics use a $\gamma=1$ sharpener that is
not applied.

From here on `A` is $X^\top$; `E_vstar_orig` (= `E_vstar`) keeps the raw rows and `A0` the raw data matrix.

In [ ]:
# Instructions steps 1-4: background whitening W, target second moment S_eta, sharpening S_eta^(gamma/2). float64, CPU.
E_vstar_orig = E_vstar                                    # raw rows (Gemma: TL-scaled), for the original fit 3.1.d'
A0 = E_vstar.T.contiguous().to(DEVICE)                    # (d, |V*|) raw

# ---- step 1: background statistics (built once per model by experiments/whitening) ----
BG_PATH = WH_DIR / f"bg_{BACKGROUND}_stats.pt"
TARGETS_PATH = WH_DIR / "targets" / f"{concept_slug}.json"
for _p, _what in [(BG_PATH, f"background statistics ({BACKGROUND})"), (TARGETS_PATH, f"target token ids for {CONCEPT!r}")]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} is missing: the {_what} must be built first (experiments/whitening)")
stats_bg = torch.load(BG_PATH, map_location="cpu")
missing_keys = {"mu", "lam", "Q"} - set(stats_bg)
assert not missing_keys, f"{BG_PATH} lacks {missing_keys}"
assert stats_bg["Q"].shape == (D_MODEL, D_MODEL), f"{BG_PATH}: Q is {tuple(stats_bg['Q'].shape)}, expected d = {D_MODEL}"
stats_bg = {k: (v.double() if torch.is_tensor(v) else v) for k, v in stats_bg.items()}
bw = BackgroundWhitener(stats_bg, EPS_REL)

lam_bg = stats_bg["lam"].clamp_min(0)
lam_desc = lam_bg.sort(descending=True).values
shrink = lam_desc / (lam_desc + bw.eps)                   # eigenvalues of W Sigma_B W
n95 = math.ceil(0.95 * len(lam_desc))
print(f"background {BACKGROUND}: n = {stats_bg.get('n', '?')} types, d = {D_MODEL}, eps = {EPS_REL:g} x mean(lam) = {bw.eps:.3e}")
print(f"W Sigma_B W ~ I: max |lam/(lam+eps) - 1| over the top 95% of directions = {(shrink[:n95] - 1).abs().max():.3e}, "
      f"overall = {(shrink - 1).abs().max():.3e} (the smallest eigenvalues are damped by eps on purpose)")
print(f"cond(Sigma_B) = {(lam_desc[0] / lam_desc[-1]).item() if lam_desc[-1] > 0 else float('inf'):.3e}   "
      f"cond(Sigma_B + eps I) = {((lam_desc[0] + bw.eps) / (lam_desc[-1] + bw.eps)).item():.3e}")

# ---- step 3: target rows, whitened ----
targets = json.loads(TARGETS_PATH.read_text())
assert TARGET in targets, f"{TARGETS_PATH} has keys {list(targets)}, not {TARGET!r}"
target_ids = sorted({int(t) for t in targets[TARGET] if 0 <= int(t) < VOCAB_SIZE} - set(special))
assert target_ids, f"{TARGETS_PATH}[{TARGET!r}] has no usable token ids"
E_target = E.index_select(0, torch.tensor(target_ids, device=E.device)).detach().double().cpu()   # (m, d), same space as E_vstar
U_target = bw.whiten(E_target)                                                                     # (m, d)
target_set, vstar_set = set(target_ids), set(vstar_ids)
_codes31 = build_token_label_codes(CONCEPT, dataset, tokenizer)
tc_ids = {t for t in vstar_ids if _codes31.get(t, 0) == 1}                                         # T_C (concept-only)
print(f"|target| = {len(target_ids)} ({TARGET})   in V*: {len(target_set & vstar_set)} / {len(vstar_ids)}   "
      f"concept-only T_C covered: {len(target_set & tc_ids)} / {len(tc_ids)}")

# ---- step 4: sharpening; T = whitening (+ sharpening) ----
sh = Sharpener(U_target, ETA, GAMMA) if GAMMA > 0 else None
sh_diag = sh if sh is not None else Sharpener(U_target, ETA, 1.0)   # gamma = 0: diagnostics only, never applied
T = Transform(bw, sh)


def transform(rows: torch.Tensor) -> torch.Tensor:
    """Rows (n, d) in the original (TL) space -> X-space: (rows - mu_B) W S_eta^(gamma/2), float32."""
    return T.forward(rows.detach().cpu()).float()


E_X = transform(E_vstar)                                  # (|V*|, d)
E_vstar_X = E_X                                           # survives 3.3.a; all analysis cells use it
A = E_X.T.contiguous().to(DEVICE)                         # (d, |V*|): 3.1.d fits on this
m_tilde = U_target.mean(0)                                # whitened target mean (the whitened shift Delta)
m_X = m_tilde if sh is None else m_tilde @ sh.Sp          # its image in X-space (= mean of transform(E_target))

# ---- checks ----
I_d = torch.eye(D_MODEL, dtype=torch.float64)
T_inv = bw.W_inv if sh is None else sh.Sm @ bw.W_inv      # row form: z = z~ @ T_inv
print(f"inverse: max |T.matrix @ (Sm @ W_inv) - I| = {(T.matrix @ T_inv - I_d).abs().max():.2e}")
print(f"row norms: raw mean {E_vstar.norm(dim=1).mean():.3f}   X-space mean {E_X.norm(dim=1).mean():.3f}   "
      f"cond(S_eta) = {(sh_diag.g[0] / sh_diag.g[-1]).item():.3e}   cond(T.matrix) = {torch.linalg.cond(T.matrix).item():.3e}")
del I_d

# ---- diagnostics: enrichment ratios ----
g_top = sh_diag.g[:10].tolist()
print(f"\nenrichment ratios g (eta = {ETA:g}; directions the target does not use sit at g = eta): top 10 = {', '.join(f'{v:.2f}' for v in g_top)}")
print(f"directions with g > 1: {int((sh_diag.g > 1).sum())} / {D_MODEL}   g > 2: {int((sh_diag.g > 2).sum())}   "
      f"stretch range applied: g^(gamma/2) in [{sh_diag.g[-1].item() ** (GAMMA / 2):.3f}, {sh_diag.g[0].item() ** (GAMMA / 2):.3f}]")

# ---- diagnostics: contrastive PCA on the whitened Wikipedia background types ----
N_CPCA, N_CPCA_TOK = 5, 15
WIKI_TYPES_PATH = WH_DIR / "bg_wiki_types.json"
if WIKI_TYPES_PATH.exists():
    wiki_ids = [int(t) for t in json.loads(WIKI_TYPES_PATH.read_text()) if 0 <= int(t) < VOCAB_SIZE]
    P5 = sh_diag.P[:, :N_CPCA]                                         # (d, 5)
    signs = torch.sign(m_tilde @ P5)                                   # orient each p_j towards the target mean
    signs[signs == 0] = 1.0
    M5 = bw.W @ (P5 * signs)                                           # whiten + project in one (d, 5) map
    scores = torch.cat([(E.index_select(0, torch.tensor(wiki_ids[i:i + 20_000], device=E.device)).detach().double().cpu()
                         - bw.mu) @ M5 for i in range(0, len(wiki_ids), 20_000)])   # (N_wiki, 5)
    cpca = {}
    for j in range(N_CPCA):
        top = torch.argsort(scores[:, j], descending=True)[:N_CPCA_TOK].tolist()
        toks = tokenizer.convert_ids_to_tokens([wiki_ids[i] for i in top])
        cpca[f"p{j + 1} (g={sh_diag.g[j]:.2f})"] = [f"{t}{' *' if wiki_ids[i] in target_set else ''}  {scores[i, j]:.2f}"
                                                   for t, i in zip(toks, top)]
    hit = {k: sum(s.split("  ")[0].endswith(" *") for s in v) for k, v in cpca.items()}
    print(f"\ncPCA: top {N_CPCA_TOK} of {len(wiki_ids)} whitened Wikipedia types on p_1..p_{N_CPCA} "
          f"(* = in the target set; per direction: {hit})")
    cpca_df = pd.DataFrame(cpca)
    cpca_df.index += 1
    display(cpca_df)
    del scores
else:
    print(f"\n{WIKI_TYPES_PATH} not found: cPCA diagnostic skipped")

print(f"\nA (transformed): {tuple(A.shape)}   A0 (raw): {tuple(A0.shape)}")

background wiki: n = 107472 types, d = 2304, eps = 0.0001 x mean(lam) = 2.712e-04
W Sigma_B W ~ I: max |lam/(lam+eps) - 1| over the top 95% of directions = 3.525e-04, overall = 2.512e-02 (the smallest eigenvalues are damped by eps on purpose)
cond(Sigma_B) = 3.314e+04   cond(Sigma_B + eps I) = 3.231e+04
|target| = 3124 (page)   in V*: 2592 / 4780   concept-only T_C covered: 1832 / 1870
inverse: max |T.matrix @ (Sm @ W_inv) - I| = 2.51e-14
row norms: raw mean 81.424   X-space mean 61.443   cond(S_eta) = 2.394e+02   cond(T.matrix) = 3.805e+02

enrichment ratios g (eta = 0.05; directions the target does not use sit at g = eta): top 10 = 14.20, 9.84, 8.65, 7.86, 7.74, 7.12, 6.44, 6.18, 5.94, 5.67
directions with g > 1: 892 / 2304   g > 2: 367   stretch range applied: g^(gamma/2) in [0.244, 3.768]

cPCA: top 15 of 107472 whitened Wikipedia types on p_1..p_5 (* = in the target set; per direction: {'p1 (g=14.20)': 15, 'p2 (g=9.84)': 2, 'p3 (g=8.65)': 15, 'p4 (g=7.86)': 12, 'p5 (g=7.74)': 14})

,p1 (g=14.20),p2 (g=9.84),p3 (g=8.65),p4 (g=7.86),p5 (g=7.74)
1,▁the * 25.37,▁թ 5.35,5 * 13.19,", * 42.35",. * 20.24
2,▁that * 22.65,▁ն 5.31,4 * 12.83,. * 39.01,", * 18.05"
3,▁this * 20.92,▁Waray 5.24,3 * 12.82,: * 32.68,: * 17.14
4,▁in * 20.19,",‎ 5.11",6 * 12.54,""" * 31.14",""" * 16.21"
5,▁with * 19.90,".""< 4.92",7 * 12.52,- * 30.61,; * 14.25
6,▁to * 19.86,▁վ 4.55,1 * 12.18,; * 29.05,- * 13.82
7,▁as * 19.66,ricio 4.45,2 * 12.16,' * 26.81,) * 12.22
8,▁for * 19.57,IÓN 4.44,8 * 12.12,) * 26.60,▁three * 11.54
9,▁they * 19.54,▁débuts 4.38,9 * 11.99,/ * 25.29,▁two * 11.45
10,▁and * 19.42,đồng 4.37,0 * 10.63,( 22.84,▁four * 11.40



A (transformed): (2304, 4780)   A0 (raw): (2304, 4780)


### 3.1.d Fit Sparse MF (alternating least squares + WTA)

The fit alternates two closed-form ridge updates:

$$Z \leftarrow A Y^\top (Y Y^\top + \lambda I)^{-1},\qquad Y \leftarrow (Z^\top Z + \lambda I)^{-1} Z^\top A$$

After each $Y$ update it applies WTA, which keeps the top-$s$ fraction of $|Y_{i,:}|$ and zeroes the rest.
It also rescales between the two factors. Initialization is $\mathcal N(0,1)$, and the fit stops early after
500 iterations without an improvement of more than $10^{-4}$. If the matrix inverse is singular,
`fit_with_ridge` retries with $\lambda \times 10$.

This is the slowest cell in Step 3.1. It prints progress every 50 iterations.

**Whitened/sharpened variant:** `A` is $X^\top$ from 3.1.c′, so `Z` and `Y` below are the transformed factorization that
everything downstream uses. The fit is cached in `RUN_DIR/smf.pt` together with its configuration; a cache made with another
configuration (or another $V^*$) is moved aside as `*.stale` with the judge/MC/generation caches, and the fit reruns.


In [ ]:
# Resume: RUN_DIR/smf.pt is loaded when its configuration matches; otherwise it (and the caches built on it) are moved aside.
SMF_PATH = RUN_DIR / "smf.pt"
fit_meta = {"model": MODEL_NAME, "vstar_ids": vstar_ids, "background": BACKGROUND, "target": TARGET, "gamma": GAMMA,
            "eta": ETA, "eps_rel": EPS_REL, "rank": RANK, "sparsity": SPARSITY, "ridge": RIDGE, "max_iter": MAX_ITER,
            "patience": PATIENCE, "seed": SEED, "n_target": len(target_ids)}
ck = torch.load(SMF_PATH, weights_only=False) if SMF_PATH.exists() else None
if ck is not None and ck.get("meta") != fit_meta:
    print(f"{SMF_PATH} was made with another configuration: re-fitting")
    invalidate(RUN_DIR, ["smf.pt", "judge.json", "mc.json", "gen.json", "summary.json"])
    ck = None
if ck is not None:
    Z, Y, ridge_used, t_fit = ck["Z"], ck["Y"], ck["ridge_used"], {"elapsed": ck["elapsed"]}
    print(f"loaded {SMF_PATH}")
else:
    set_seed(SEED)
    smf = SparseMatrixFactorization(RANK, fitting_device=DEVICE, g_sparsity=SPARSITY)
    with Timer() as t_fit:
        ridge_used = fit_with_ridge(smf, A, MAX_ITER, patience=PATIENCE, base_reg=RIDGE)

    Z = smf.F_.detach().float().cpu()      # (d, k)      X-space features z~_i (columns)
    Y = smf.G_.detach().float().cpu().T    # (k, |V*|)   sparse coefficients
    torch.save({"Z": Z, "Y": Y, "ridge_used": ridge_used, "elapsed": t_fit["elapsed"], "meta": fit_meta}, SMF_PATH)
print(f"\nfitted in {t_fit['elapsed']:.1f}s (lambda={ridge_used:g})  Z: {tuple(Z.shape)}  Y: {tuple(Y.shape)}")

loaded runs/whiten/gemma/Harry_Potter/wiki_page_g1/smf.pt

fitted in 16.3s (lambda=0.0001)  Z: (2304, 100)  Y: (100, 4780)


### 3.1.d′ Reference fit on the original (untransformed) matrix

Same class, settings and seed, on the raw $A_0 = E_{V^*}^\top$. `Z0` and `Y0` are used only for comparison: 3.1.e/f (errors),
3.2.e′ (the raw fit's concept features `F_C0`) and 3.2.w (feature matching). Step 3.3 does not use them. The fit does not depend
on the whitening configuration, so it is cached once per notebook in `ORIG_DIR/smf0.pt` and shared by every configuration.

In [ ]:
SMF0_PATH = ORIG_DIR / "smf0.pt"
fit0_meta = {"model": MODEL_NAME, "vstar_ids": vstar_ids, "rank": RANK, "sparsity": SPARSITY, "ridge": RIDGE,
             "max_iter": MAX_ITER, "patience": PATIENCE, "seed": SEED}
ck0 = torch.load(SMF0_PATH, weights_only=False) if SMF0_PATH.exists() else None
if ck0 is not None and ck0.get("meta") != fit0_meta:
    print(f"{SMF0_PATH} was made with another configuration: re-fitting")
    invalidate(ORIG_DIR, ["smf0.pt", "judge0.json"])
    ck0 = None
if ck0 is not None:
    Z0, Y0, ridge_used0, t_fit0 = ck0["Z"], ck0["Y"], ck0["ridge_used"], {"elapsed": ck0["elapsed"]}
    print(f"loaded {SMF0_PATH}")
else:
    set_seed(SEED)
    smf0 = SparseMatrixFactorization(RANK, fitting_device=DEVICE, g_sparsity=SPARSITY)
    with Timer() as t_fit0:
        ridge_used0 = fit_with_ridge(smf0, A0, MAX_ITER, patience=PATIENCE, base_reg=RIDGE)

    Z0 = smf0.F_.detach().float().cpu()      # (d, k)      raw-space features
    Y0 = smf0.G_.detach().float().cpu().T    # (k, |V*|)   sparse coefficients
    torch.save({"Z": Z0, "Y": Y0, "ridge_used": ridge_used0, "elapsed": t_fit0["elapsed"], "meta": fit0_meta}, SMF0_PATH)
print(f"\nfitted in {t_fit0['elapsed']:.1f}s (lambda={ridge_used0:g})  Z0: {tuple(Z0.shape)}  Y0: {tuple(Y0.shape)}")

loaded runs/whiten/gemma/Harry_Potter/original/smf0.pt

fitted in 16.7s (lambda=0.0001)  Z0: (2304, 100)  Y0: (100, 4780)


### 3.1.e Check the factorization

Reconstruction quality and sparsity. Each feature should use ≈ $\lceil s|V^*|\rceil$ tokens. Tokens that
no feature covers are left as pure reconstruction error $\boldsymbol\varepsilon_t$.

Both fits are reported, each against its own data matrix. The two errors are not comparable in absolute terms: whitening
flattens the spectrum, so the transformed error is higher for any rank-$k$ model (the SVD baseline in 3.1.f shows by how much).

The second cell is the optional check of Instructions step 6: $|\cos(\tilde{\mathbf z}_i, \tilde{\mathbf m})|$ between each fitted factor
and the target mean in X-space ($\tilde{\mathbf m} = $ `m_X`). Concept factors should be among the highest; low values flag generic factors.


In [ ]:
A_cpu, A0_cpu = A.cpu(), A0.cpu()
rel_err = (torch.norm(A_cpu - Z @ Y) / torch.norm(A_cpu)).item()        # transformed fit, X-space
rel_err0 = (torch.norm(A0_cpu - Z0 @ Y0) / torch.norm(A0_cpu)).item()   # original fit, raw space
nz = Y.abs() > 0
nz0 = Y0.abs() > 0

for name, err, mask in [("transformed", rel_err, nz), ("original", rel_err0, nz0)]:
    tokens_per_feature, features_per_token = mask.sum(1), mask.sum(0)
    print(f"[{name}] relative reconstruction error ||A - ZY||_F / ||A||_F = {err:.4f}")
    print(f"[{name}] tokens per feature : min={tokens_per_feature.min().item()}  max={tokens_per_feature.max().item()}")
    print(f"[{name}] features per token : mean={features_per_token.float().mean():.2f}  "
          f"max={features_per_token.max().item()}  uncovered tokens={int((features_per_token == 0).sum())}\n")

[transformed] relative reconstruction error ||A - ZY||_F / ||A||_F = 0.8578
[transformed] tokens per feature : min=48  max=48
[transformed] features per token : mean=1.00  max=10  uncovered tokens=2165

[original] relative reconstruction error ||A - ZY||_F / ||A||_F = 0.9158
[original] tokens per feature : min=48  max=48
[original] features per token : mean=1.00  max=5  uncovered tokens=1394



In [ ]:
# Instructions step 6 (optional check): |cos| of each X-space factor with the target mean m_X.
cos_m = F.cosine_similarity(Z.double(), m_X.unsqueeze(1), dim=0).abs()       # (k,)
cos_m_order = torch.argsort(cos_m, descending=True).tolist()
print(f"|cos(z~_i, m_X)|: median={cos_m.median():.3f}  max={cos_m.max():.3f}  "
      f"(random-direction scale ~ {math.sqrt(2 / (math.pi * D_MODEL)):.3f})")
cos_m_df = pd.DataFrame({
    "feature": cos_m_order[:10],
    "|cos(z~, m_X)|": cos_m[cos_m_order[:10]].numpy(),
    "top tokens": [", ".join(tokenizer.convert_ids_to_tokens(
        [vstar_ids[j] for j in torch.argsort(Y[f].abs(), descending=True)[:8].tolist()])) for f in cos_m_order[:10]],
})
cos_m_df

|cos(z~_i, m_X)|: median=0.219  max=0.446  (random-direction scale ~ 0.017)


,feature,"|cos(z~, m_X)|",top tokens
0,94,0.446372,"▁within, ▁in, ▁with, ▁among, ▁during, ▁from, ▁alongside, ▁at"
1,93,0.424599,"▁can, ▁will, ▁may, ▁is, ▁might, ▁could, ▁would, ▁should"
2,60,0.420947,"▁high, high, ▁High, ▁low, ▁great, ▁big, ▁strong, ▁long"
3,13,0.410595,"▁making, ▁make, ▁makes, ▁made, made, ▁mak, ▁giving, ▁taking"
4,15,0.400586,"▁a, ▁an, ▁another, a, ▁something, ▁some, An, ▁A"
5,41,0.389360,"▁we, ▁they, ▁she, ▁he, ▁you, ▁there, ▁it, ▁I"
6,19,0.366773,"▁principal, ▁main, ▁key, ▁primary, ▁major, ▁leading, ▁lead, ▁important"
7,16,0.364361,"▁name, ▁word, ▁names, ▁named, ▁words, ▁term, ▁title, ▁terms"
8,59,0.359856,"▁way, means, ▁ways, ▁mean, ▁meaning, ▁manner, way, ▁meant"
9,1,0.359030,"▁previous, ▁previously, ▁originally, ▁original, ▁earlier, ▁initially, ▁initial, ▁last"


In [ ]:
print(Z.size())
print(Y.size())
print(RANK)

torch.Size([2304, 100])
torch.Size([100, 4780])
100


### 3.1.f Baseline: truncated SVD with the same $k$

The rank-$k$ truncated SVD $A_k = U_k \Sigma_k V_k^\top$ is the best possible rank-$k$ approximation
(Eckart–Young theorem), so no rank-$k$ factorization can have a lower error, including Sparse MF.
The gap between the two errors is what the WTA sparsity costs in reconstruction.

In [ ]:
for name, A_mat, err, mask, Z_mat in [("transformed", A_cpu, rel_err, nz, Z), ("original", A0_cpu, rel_err0, nz0, Z0)]:
    U, S, Vh = torch.linalg.svd(A_mat, full_matrices=False)
    A_svd = U[:, :RANK] @ torch.diag(S[:RANK]) @ Vh[:RANK]
    svd_rel_err = (torch.norm(A_mat - A_svd) / torch.norm(A_mat)).item()

    svd_nnz = U[:, :RANK].numel() + RANK + Vh[:RANK].numel()
    smf_nnz = Z_mat.numel() + int(mask.sum())
    print(f"[{name}] rank-{RANK} truncated SVD : relative error = {svd_rel_err:.4f}   (parameters: {svd_nnz:,})")
    print(f"[{name}] rank-{RANK} Sparse MF     : relative error = {err:.4f}   (nonzero parameters: {smf_nnz:,})")

    # Smallest SVD rank whose error is no worse than Sparse MF's.
    tail = torch.sqrt(torch.flip(torch.cumsum(torch.flip(S ** 2, [0]), 0), [0]) / (S ** 2).sum())
    matching_rank = int((tail <= err).nonzero()[0]) if (tail <= err).any() else len(S)
    print(f"[{name}] SVD reaches the Sparse MF error ({err:.4f}) at rank {matching_rank}\n")
del U, S, Vh, A_svd

[transformed] rank-100 truncated SVD : relative error = 0.7917   (parameters: 708,500)
[transformed] rank-100 Sparse MF     : relative error = 0.8578   (nonzero parameters: 235,200)
[transformed] SVD reaches the Sparse MF error (0.8578) at rank 49

[original] rank-100 truncated SVD : relative error = 0.8219   (parameters: 708,500)
[original] rank-100 Sparse MF     : relative error = 0.9158   (nonzero parameters: 235,200)
[original] SVD reaches the Sparse MF error (0.9158) at rank 5



---
# Step 3.2: Identifying concept-related features

Most of the $k$ features are general (punctuation, function words, and so on). The selection runs in two filters:

1. **Mass ratio** $\rho_i$: is feature $i$ used mostly by concept tokens rather than neutral tokens? Keep the feature if $\rho_i \ge \tau = 2.0$.
2. **LLM judge** (Gemini, two stages): describe the feature's tokens *without* seeing the concept name,
   then ask whether that description is the concept. Keep the feature if `is_member` is true and confidence ≥ 0.85.

The features that pass both filters form $F_C$.

### 3.2.a Label every token in $V^*$

$\ell_t \in$ {concept, neutral, both}, depending on whether $t$ appears in $S_C$ only, $S_N$ only, or both.
Codes: `1` = concept, `0` = neutral, `2` = both.

In [ ]:
token_to_code = build_token_label_codes(CONCEPT, dataset, tokenizer)
codes = np.array([token_to_code.get(t, 0) for t in vstar_ids])

print(f"T_C (concept-only) = {(codes == 1).sum()}   T_N (neutral-only) = {(codes == 0).sum()}   "
      f"both = {(codes == 2).sum()}")

T_C (concept-only) = 1870   T_N (neutral-only) = 2229   both = 681


### 3.2.b Score features by mass ratio and keep $\rho_i \ge \tau$

As implemented in `compute_embedding_stats` (column `ratio_abs`):

$$\rho_i = \frac{\operatorname{mean}_{t\in T_C} |Y_{i,t}|}{\operatorname{mean}_{t\in T_N} |Y_{i,t}|}$$

Tokens labeled *both* are excluded from the ratio. The threshold sweep below is the per-concept view of
Figure 6 in the paper.

In [ ]:
stats = compute_embedding_stats(Y.T, vstar_ids, token_to_code, CONCEPT).sort_values("ratio_abs", ascending=False)

for thr in [1.0, 1.25, 1.5, 1.75, 2.0, 2.25]:
    print(f"rho >= {thr:<4}: {(stats['ratio_abs'] >= thr).sum():3d} / {RANK} features")

candidates = stats[stats["ratio_abs"] >= TAU]
print(f"\n{len(candidates)} candidate features pass tau = {TAU}")
candidates[["feature", "ratio_abs", "mean_abs_concept", "mean_abs_neutral", "num_concept", "num_neutral", "num_both"]]

rho >= 1.0 :  83 / 100 features
rho >= 1.25:  76 / 100 features
rho >= 1.5 :  70 / 100 features
rho >= 1.75:  65 / 100 features
rho >= 2.0 :  61 / 100 features
rho >= 2.25:  56 / 100 features

61 candidate features pass tau = 2.0


,feature,ratio_abs,mean_abs_concept,mean_abs_neutral,num_concept,num_neutral,num_both
85,85,125.073794,0.003143,0.000025,45,1,2
46,46,51.452493,0.002624,0.000051,44,2,2
98,98,28.422252,0.002727,0.000096,40,2,6
44,44,22.760720,0.002197,0.000097,38,4,6
7,7,16.366878,0.002062,0.000126,37,5,6
...,...,...,...,...,...,...,...
75,75,2.194331,0.001753,0.000799,22,18,8
42,42,2.154477,0.001544,0.000716,26,15,7
1,1,2.113098,0.001504,0.000712,21,13,14
2,2,2.010309,0.001102,0.000548,25,15,8


### 3.2.c Collect the tokens the judge will see

For each candidate, the code takes the tokens with nonzero $Y_{i,t}$, sorted by $|Y_{i,t}|$, and keeps the top 20.
The paper text says "concept-labeled tokens", but the released code sends the top activating tokens
whatever their label. This notebook follows the code. The `labels` column shows each token's label.

In [ ]:
token_label_map = {tid: "both" if c == 2 else CONCEPT if c == 1 else "Neutral" for tid, c in token_to_code.items()}
feature_rows = collect_feature_rows_for_embeddings(
    Y.T, vstar_ids, token_label_map, tl_model, _safe_model_name(MODEL_NAME), CONCEPT, RANK,
)
feat_tokens = pd.DataFrame(feature_rows).set_index("feature")[["activating_tokens", "labels"]]

cand = candidates[["feature", "ratio_abs"]].join(feat_tokens, on="feature").reset_index(drop=True)
if SMOKE:
    cand = cand.head(3)  # smoke: judge only the 3 highest-rho candidates
from ember.judge_words import desc_sys_prompt, judge_cache_name, judge_items, token_word_counts

DESC_SYS = desc_sys_prompt(JUDGE_FORMAT, multilingual=False)   # = TOKENS_DESC_SYS for "tokens"
jw_counts = token_word_counts(tokenizer, [s for s, _ in dataset]) if JUDGE_FORMAT == "words" else {}


def judge_input(Ymat: torch.Tensor, f: int, activating_tokens) -> list:
    """What Stage 1 sees for feature f of Ymat: "tokens" = the top activating tokens (as before); "words" = token → source words."""
    if JUDGE_FORMAT == "tokens":
        return list(activating_tokens[:TOP_K_TOKENS])
    mags = Ymat[f].abs()
    top = torch.argsort(mags, descending=True)[:TOP_K_TOKENS]
    top = top[mags[top] > 0].tolist()
    return judge_items([vstar_ids[j] for j in top], lambda t: tokenizer.decode([t]), jw_counts, JUDGE_FORMAT,
                       multilingual=False)


cand["judge_tokens"] = [judge_input(Y, int(f), toks) for f, toks in zip(cand["feature"], cand["activating_tokens"])]
cand[["feature", "ratio_abs", "judge_tokens"]]

### 3.2.d-0 Rate-limited Gemini client

The free tier allows **15 requests per minute per model**. Stage 1 and Stage 2 make one request per
candidate each, so 2 × (number of candidates) requests in total. This wrapper does three things:

- spaces requests at least `60 / RPM` seconds apart (with a 10% margin);
- on a per-minute 429, waits the `retry in Ns` delay the server reports, then retries;
- fails immediately on a *daily* quota error, because waiting would not help.

Answers are cached in `descriptions` and `verdicts`. Re-running a stage only queries the features that
are still missing, so a crash never wastes quota you already spent.

In [ ]:
import re
import time

GEMINI_RPM = 15


class ThrottledGemini:
    def __init__(self, inner: SimpleGeminiClient, rpm: int = GEMINI_RPM, max_attempts: int = 6):
        inner.max_retries = 1  # retries are handled here, with the server's retry delay
        self.inner = inner
        self.interval = 60.0 / rpm * 1.1
        self.max_attempts = max_attempts
        self._last = 0.0

    def generate(self, prompt: str) -> str:
        for attempt in range(self.max_attempts):
            time.sleep(max(0.0, self._last + self.interval - time.monotonic()))
            self._last = time.monotonic()
            try:
                return self.inner.generate(prompt)
            except RuntimeError as e:
                msg = str(e)
                if "429" not in msg or "PerDay" in msg or attempt == self.max_attempts - 1:
                    raise
                m = re.search(r"retry in ([\d.]+)s", msg)
                time.sleep(float(m.group(1)) + 1 if m else 60)


USE_THROTTLE = False  # billing enabled; set True to go back to free-tier pacing

if USE_THROTTLE:
    client = ThrottledGemini(SimpleGeminiClient(model_name=GEMINI_MODEL))
else:
    client = SimpleGeminiClient(model_name=GEMINI_MODEL)

# Persisted in RUN_DIR/judge.json, saved after every Gemini call, so a restart re-queries nothing.
# 3.1.d moves judge.json aside whenever it re-fits.
JUDGE_PATH = RUN_DIR / judge_cache_name(JUDGE_FORMAT, "tokens")   # judge.json for the original format
_judge = load_json(JUDGE_PATH, {})
descriptions = {int(f): d for f, d in _judge.get("descriptions", {}).items()}
verdicts = {int(f): tuple(v) for f, v in _judge.get("verdicts", {}).items()}


def save_judge() -> None:
    save_json(JUDGE_PATH, {"descriptions": descriptions, "verdicts": verdicts})

print(f"{2 * len(cand)} requests needed, {len(descriptions)} descriptions and {len(verdicts)} verdicts cached; "
      f"throttle={'on' if USE_THROTTLE else 'off'}")

### 3.2.d LLM judge, stage 1: describe each feature *without* the concept name

Stage 1 never sees the concept name. This avoids confirmation bias, where tokens like *wizard* could
look like "Harry Potter" just because the judge was told the concept.

In [ ]:
print("---- Stage 1 prompt ----")
print(TOKENS_DESC_SYS.strip(), "\n\nTokens: {tokens}\nExplanation of feature behavior:")

todo = cand[~cand["feature"].astype(int).isin(descriptions)]
for _, row in tqdm(todo.iterrows(), total=len(todo), desc="Stage 1: describe"):
    descriptions[int(row["feature"])] = get_description(client, row["judge_tokens"], sys_prompt=DESC_SYS)
    save_judge()
cand["description"] = cand["feature"].map(descriptions)
cand[["feature", "ratio_abs", "description"]]

In [ ]:
# Magnitude of each whitened + sharpened (X-space) feature direction ||z~_i|| (column of Z), relative to all k
# features, sorted largest first:
#   share      = ||z~_i|| / sum_j ||z~_j||       (fraction of the total magnitude)
#   percentile = % of the k features with ||z~_j|| <= ||z~_i||   (100 = largest)
z_norms = Z.norm(dim=0)                                                     # (k,)
z_share = (z_norms / z_norms.sum()).numpy()
z_pct = ((z_norms[None, :] <= z_norms[:, None]).float().mean(1) * 100).numpy()

feat_ids = cand["feature"].astype(int)
cand["||z||"] = z_norms.numpy()[feat_ids]
cand["||z|| share %"] = z_share[feat_ids] * 100
cand["||z|| percentile"] = z_pct[feat_ids]

print(f"||z~|| over all {Z.shape[1]} X-space features: total={z_norms.sum():.1f}  mean={z_norms.mean():.2f}  "
      f"min={z_norms.min():.2f}  max={z_norms.max():.2f}  (uniform share = {100 / Z.shape[1]:.2f}%)")
cand.sort_values("||z||", ascending=False)[
    ["feature", "ratio_abs", "||z||", "||z|| share %", "||z|| percentile", "description"]
].style.format({"ratio_abs": "{:.2f}", "||z||": "{:.2f}", "||z|| share %": "{:.2f}%", "||z|| percentile": "{:.0f}"})


||z~|| over all 100 X-space features: total=22284.5  mean=222.85  min=155.91  max=520.10  (uniform share = 1.00%)


,feature,ratio_abs,||z||,||z|| share %,||z|| percentile,description
29,64,3.87,345.57,1.55%,97,This feature vector represents grammatical determiners and articles used to specify or introduce nouns.
23,93,4.54,323.90,1.45%,96,"Auxiliary and modal verbs functioning as predicates to express tense, mood, or aspect."
31,28,3.70,313.10,1.41%,92,This vector represents small integer numbers and quantifiers.
50,95,2.40,252.54,1.13%,82,This vector activates for masculine singular personal pronouns and morphological suffixes common in proper names.
21,38,4.59,251.09,1.13%,80,"These tokens function as conjunctive adverbs and transition words used to connect ideas, indicate contrast, consequence, or continuation between clauses."
49,0,2.56,247.60,1.11%,78,This feature vector represents units of time and temporal duration.
32,87,3.60,241.39,1.08%,76,"These tokens represent quantifying adjectives and determiners indicating quantity, amount, or diversity."
1,46,51.45,232.44,1.04%,73,"This feature vector represents the concepts of magic, fantasy, and witchcraft."
41,90,2.94,232.41,1.04%,72,"This vector represents words used to introduce causal relationships, justifications, or logical conditions."
25,6,4.53,230.82,1.04%,71,"This vector represents adverbs of frequency, degree, and general qualification indicating typicality or conditionality."


In [ ]:
N_SHOW = 10
LABEL_TAG = {1: "C", 0: "N", 2: "B"}  # concept / neutral / both

for _, row in cand.sort_values("ratio_abs", ascending=False).iterrows():
    feat = int(row["feature"])
    mags = Y[feat].abs()
    top = torch.argsort(mags, descending=True)[:N_SHOW]
    top = top[mags[top] > 0].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in top])

    print(f"=== feature {feat}  rho={row['ratio_abs']:.2f} ===")
    print(f"  Gemini : {descriptions.get(feat, '<not described yet>')}")
    print("  tokens : " + ", ".join(f"{t!r}[{LABEL_TAG[codes[j]]}] {mags[j]:.2f}" for t, j in zip(toks, top)))
    print()

=== feature 85  rho=125.07 ===
  Gemini : This feature activates on names, terms, and locations specifically associated with the Harry Potter universe.
  tokens : 'ffindor'[C] 0.35, 'therin'[C] 0.29, 'umbledore'[C] 0.25, '▁Dumbledore'[C] 0.22, '▁Weasley'[C] 0.21, '▁Hogwarts'[C] 0.20, '▁Malfoy'[C] 0.20, '▁Rowling'[C] 0.20, '▁Potter'[C] 0.20, '▁Hermione'[C] 0.19

=== feature 46  rho=51.45 ===
  Gemini : This feature vector represents the concepts of magic, fantasy, and witchcraft.
  tokens : '▁magic'[C] 0.36, 'magic'[C] 0.34, 'Magic'[C] 0.34, '▁Magic'[B] 0.33, '▁magical'[C] 0.27, 'magical'[C] 0.26, '▁Magical'[C] 0.23, '▁wizard'[C] 0.19, '▁wizards'[C] 0.18, '▁Wizard'[C] 0.16

=== feature 98  rho=28.42 ===
  Gemini : This feature vector represents narrative literature and storytelling concepts.
  tokens : '▁stories'[B] 0.30, '▁tales'[C] 0.29, '▁story'[B] 0.28, '▁fiction'[C] 0.25, '▁Tales'[C] 0.22, '▁novel'[B] 0.20, 'fiction'[C] 0.19, '▁novels'[C] 0.19, '▁narrative'[C] 0.18, '▁fictional'[C]

### 3.2.e LLM judge, stage 2: is the description *the* concept?

Stage 2 sees only the concept name and the Stage 1 description. It returns `{"is_member", "confidence"}`.
A feature joins $F_C$ only if `is_member` is true **and** confidence ≥ 0.85.

In [ ]:
print("---- Stage 2 prompt ----")
print(MEMBERSHIP_PROMPT.format(concept=CONCEPT, desc="{description}").strip())

todo = {f: descriptions[f] for f in cand["feature"].astype(int).tolist() if f in descriptions and f not in verdicts}
for feat, desc in tqdm(todo.items(), desc="Stage 2: classify"):
    verdicts[feat] = classify_membership_with_desc(client, CONCEPT, desc)
    save_judge()

cand["is_member"] = cand["feature"].map(lambda f: verdicts[int(f)][0])
cand["confidence"] = pd.to_numeric(cand["feature"].map(lambda f: verdicts[int(f)][1]), errors="coerce").fillna(0.0)
cand["selected"] = (cand["is_member"] == True) & (cand["confidence"] >= CONFIDENCE)

F_C = sorted(cand.loc[cand["selected"], "feature"].astype(int).tolist())
print(f"\nF_C = {F_C}  ({len(F_C)} concept-related features)")
cand[["feature", "ratio_abs", "is_member", "confidence", "selected", "description"]]

---- Stage 2 prompt ----
You get a concept name and a feature's description.
Decide if this feature describes the given concept. Consider if the feature is DISTINCTIVE for the concept (not broad like "sport" for "basketball") and not too noisy.
A feature should be marked true only if the concept is central and dominant in the description, not just one of several themes.
Return ONLY JSON: {"is_member": true|false, "confidence": 0..1}

Example 1:
Concept: Charity
Description: This vector represents elements related to giving and donations to charitable causes.
Answer: {"is_member": true, "confidence": 0.98}

Example 2:
Concept: Charity
Description: This vector represents elements related to money, such as gambling, investing and charity.
Answer: {"is_member": false, "confidence": 0.80}

Concept: Harry Potter
Description: {description}
Answer:


Stage 2: classify: 0it [00:00, ?it/s]


F_C = [85]  (1 concept-related features)


,feature,ratio_abs,is_member,confidence,selected,description
0,85,125.073794,True,1.00,True,"This feature activates on names, terms, and locations specifically associated with the Harry Potter universe."
1,46,51.452493,False,0.85,False,"This feature vector represents the concepts of magic, fantasy, and witchcraft."
2,98,28.422252,False,0.95,False,This feature vector represents narrative literature and storytelling concepts.
3,44,22.760720,False,1.00,False,"This feature vector represents the action and practice of acquiring, processing, and communicating knowledge or language."
4,7,16.366878,False,0.99,False,"This vector represents concepts related to visual perception, observation, and personal viewpoints or opinions."
...,...,...,...,...,...,...
56,75,2.194331,False,0.99,False,"This vector is related to journalism, news media, and publishing organizations or outlets."
57,42,2.154477,False,0.99,False,This feature vector represents color terms and chromatic descriptors.
58,1,2.113098,False,0.99,False,"This vector represents temporal sequencing, precedence, and past occurrences."
59,2,2.010309,False,0.99,False,"This feature vector represents concepts of superiority, priority, and maximal ranking."


### 3.2.e′ Same selection on the original fit → $F_{C0}$

The mass ratio ($\tau$) and the two-stage judge (same client, prompts and thresholds) applied to `Y0`, so the concept directions of
both fits are known. Answers are cached in `ORIG_DIR/judge0.json` (saved after every call), shared by all configurations.
The main flow (3.2.f and Step 3.3) uses the transformed fit's $F_C$. The last line lists $|\cos(\tilde{\mathbf z}_i, \tilde{\mathbf m})|$ for $i\in F_C$.

In [ ]:
# tau filter + 2-stage judge on the original fit (Y0); cached in ORIG_DIR/judge0.json.
JUDGE0_PATH = ORIG_DIR / judge_cache_name(JUDGE_FORMAT, "tokens", "judge0")
_judge0 = load_json(JUDGE0_PATH, {})
descriptions0 = {int(f): d for f, d in _judge0.get("descriptions", {}).items()}
verdicts0 = {int(f): tuple(v) for f, v in _judge0.get("verdicts", {}).items()}


def save_judge0() -> None:
    save_json(JUDGE0_PATH, {"descriptions": descriptions0, "verdicts": verdicts0})


stats0 = compute_embedding_stats(Y0.T, vstar_ids, token_to_code, CONCEPT).sort_values("ratio_abs", ascending=False)
feat_tokens0 = pd.DataFrame(collect_feature_rows_for_embeddings(
    Y0.T, vstar_ids, token_label_map, tl_model, _safe_model_name(MODEL_NAME), CONCEPT, RANK,
)).set_index("feature")[["activating_tokens", "labels"]]
cand0 = stats0[stats0["ratio_abs"] >= TAU][["feature", "ratio_abs"]].join(feat_tokens0, on="feature").reset_index(drop=True)
if SMOKE:
    cand0 = cand0.head(3)  # smoke: judge only the 3 highest-rho candidates

for _, row in tqdm(cand0.iterrows(), total=len(cand0), desc="original fit: judge"):
    f = int(row["feature"])
    if f not in descriptions0:
        descriptions0[f] = get_description(client, judge_input(Y0, f, row["activating_tokens"]), sys_prompt=DESC_SYS)
        save_judge0()
    if f not in verdicts0:
        verdicts0[f] = classify_membership_with_desc(client, CONCEPT, descriptions0[f])
        save_judge0()

cand0["description"] = cand0["feature"].map(lambda f: descriptions0[int(f)])
cand0["is_member"] = cand0["feature"].map(lambda f: verdicts0[int(f)][0])
cand0["confidence"] = pd.to_numeric(cand0["feature"].map(lambda f: verdicts0[int(f)][1]), errors="coerce").fillna(0.0)
cand0["selected"] = (cand0["is_member"] == True) & (cand0["confidence"] >= CONFIDENCE)

F_C0 = sorted(cand0.loc[cand0["selected"], "feature"].astype(int).tolist())
print(f"original fit: {len(cand0)} candidates pass tau = {TAU}  ->  F_C0 = {F_C0}   (transformed fit: F_C = {F_C})")
print("|cos(z~_i, m_X)| for i in F_C: " + (", ".join(
    f"{f}: {cos_m[f]:.3f} (rank {cos_m_order.index(f) + 1}/{RANK})" for f in F_C) or "F_C is empty"))
cand0[["feature", "ratio_abs", "is_member", "confidence", "selected", "description"]]

In [ ]:
is_true = cand["is_member"] == True
ordered = cand.assign(
    _group=(~is_true).astype(int),
    _key=np.where(is_true, -cand["confidence"], cand["confidence"]),
).sort_values(["_group", "_key"])

for _, row in ordered.iterrows():
    feat = int(row["feature"])
    mags = Y[feat].abs()
    top = torch.argsort(mags, descending=True)[:N_SHOW]
    top = top[mags[top] > 0].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in top])

    is_member, conf = verdicts.get(feat, (None, None))
    verdict = "<not classified yet>" if feat not in verdicts else (
        f"is_member={is_member}  confidence={conf if conf is not None else 'n/a'}  "
        f"-> {'SELECTED' if row['selected'] else 'rejected'}"
    )

    print(f"=== feature {feat}  rho={row['ratio_abs']:.2f} ===")
    print(f"  Gemini  : {descriptions.get(feat, '<not described yet>')}")
    print(f"  verdict : {verdict}")
    print("  tokens  : " + ", ".join(f"{t!r}[{LABEL_TAG[codes[j]]}] {mags[j]:.2f}" for t, j in zip(toks, top)))
    print()

=== feature 85  rho=125.07 ===
  Gemini  : This feature activates on names, terms, and locations specifically associated with the Harry Potter universe.
  verdict : is_member=True  confidence=1.0  -> SELECTED
  tokens  : 'ffindor'[C] 0.35, 'therin'[C] 0.29, 'umbledore'[C] 0.25, '▁Dumbledore'[C] 0.22, '▁Weasley'[C] 0.21, '▁Hogwarts'[C] 0.20, '▁Malfoy'[C] 0.20, '▁Rowling'[C] 0.20, '▁Potter'[C] 0.20, '▁Hermione'[C] 0.19

=== feature 46  rho=51.45 ===
  Gemini  : This feature vector represents the concepts of magic, fantasy, and witchcraft.
  verdict : is_member=False  confidence=0.85  -> rejected
  tokens  : '▁magic'[C] 0.36, 'magic'[C] 0.34, 'Magic'[C] 0.34, '▁Magic'[B] 0.33, '▁magical'[C] 0.27, 'magical'[C] 0.26, '▁Magical'[C] 0.23, '▁wizard'[C] 0.19, '▁wizards'[C] 0.18, '▁Wizard'[C] 0.16

=== feature 98  rho=28.42 ===
  Gemini  : This feature vector represents narrative literature and storytelling concepts.
  verdict : is_member=False  confidence=0.95  -> rejected
  tokens  : '▁stories

In [ ]:
members = cand[cand["is_member"] == True].sort_values("ratio_abs", ascending=False)
print(f"{len(members)} / {len(cand)} candidates have is_member=True\n")

for _, row in members.iterrows():
    feat = int(row["feature"])
    mags = Y[feat].abs()
    top = torch.argsort(mags, descending=True)[:N_SHOW]
    top = top[mags[top] > 0].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in top])

    print(f"=== feature {feat}  rho={row['ratio_abs']:.2f} ===")
    print(f"  Gemini  : {row['description']}")
    print(f"  verdict : confidence={row['confidence']:.2f}  -> {'SELECTED' if row['selected'] else 'rejected (confidence < ' + str(CONFIDENCE) + ')'}")
    print("  tokens  : " + ", ".join(f"{t!r}[{LABEL_TAG[codes[j]]}] {mags[j]:.2f}" for t, j in zip(toks, top)))
    print()

1 / 61 candidates have is_member=True

=== feature 85  rho=125.07 ===
  Gemini  : This feature activates on names, terms, and locations specifically associated with the Harry Potter universe.
  verdict : confidence=1.00  -> SELECTED
  tokens  : 'ffindor'[C] 0.35, 'therin'[C] 0.29, 'umbledore'[C] 0.25, '▁Dumbledore'[C] 0.22, '▁Weasley'[C] 0.21, '▁Hogwarts'[C] 0.20, '▁Malfoy'[C] 0.20, '▁Rowling'[C] 0.20, '▁Potter'[C] 0.20, '▁Hermione'[C] 0.19



### 3.2.f Inspect the selected features

Table 3 of the paper reports only 1–3 embedding features per concept (Harry Potter on Gemma: 1).
Each selected feature should be a tight cluster of concept-specific tokens.

In [ ]:
if not F_C:
    print("No feature passed both filters: Step 3.3 will be a no-op for this concept.")

for feat in F_C:
    row = cand.loc[cand["feature"] == feat].iloc[0]
    print(f"=== feature {feat}  rho={row['ratio_abs']:.2f}  conf={row['confidence']:.2f} ===")
    print(f"  {row['description']}")
    print("  tokens:", ", ".join(f"{t!r}[{l[0]}]" for t, l in zip(row["activating_tokens"][:30], row["labels"][:30])))
    print()

=== feature 85  rho=125.07  conf=1.00 ===
  This feature activates on names, terms, and locations specifically associated with the Harry Potter universe.
  tokens: 'ffindor'[H], 'therin'[H], 'umbledore'[H], ' Dumbledore'[H], ' Weasley'[H], ' Hogwarts'[H], ' Malfoy'[H], ' Rowling'[H], ' Potter'[H], ' Hermione'[H], 'demort'[H], 'Hermione'[H], ' Voldemort'[H], ' Harry'[H], 'Potter'[H], ' Snape'[H], 'Harry'[H], ' Severus'[H], ' Draco'[H], 'Hog'[H], ' Gry'[H], ' Ginny'[H], ' Hog'[H], 'otts'[H], 'warts'[H], ' Lupin'[H], ' Remus'[H], ' Granger'[H], ' Sly'[H], ' JK'[H]



In [ ]:
# One column per selected feature: every token with nonzero Y_{i,t}, sorted by |Y_{i,t}|.
columns = {}
for feat in F_C:
    mags = Y[feat].abs()
    nz_idx = torch.nonzero(mags > 0).flatten()
    nz_idx = nz_idx[torch.argsort(mags[nz_idx], descending=True)].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in nz_idx])
    columns[f"feature {feat}"] = pd.Series(
        [f"{t} [{LABEL_TAG[codes[j]]}] {mags[j]:.2f}" for t, j in zip(toks, nz_idx)]
    )

wide = pd.DataFrame(columns).fillna("")
wide.index = range(1, len(wide) + 1)

all_cols = sorted({j for feat in F_C for j in torch.nonzero(Y[feat].abs() > 0).flatten().tolist()})
print(f"{len(F_C)} concept features, {len(all_cols)} distinct tokens "
      f"(C={sum(codes[j] == 1 for j in all_cols)}, N={sum(codes[j] == 0 for j in all_cols)}, "
      f"B={sum(codes[j] == 2 for j in all_cols)})")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(wide)

1 concept features, 48 distinct tokens (C=45, N=1, B=2)


,feature 85
1,ffindor [C] 0.35
2,therin [C] 0.29
3,umbledore [C] 0.25
4,▁Dumbledore [C] 0.22
5,▁Weasley [C] 0.21
6,▁Hogwarts [C] 0.20
7,▁Malfoy [C] 0.20
8,▁Rowling [C] 0.20
9,▁Potter [C] 0.20
10,▁Hermione [C] 0.19


In [ ]:
# # Same table as above, with token / label / |Y| in separate sub-columns, plus "words": the whole words in S_C ∪ S_N
# that each token was cut from (most frequent first), so truncated sub-word tokens like 'umbledore' can be read back.
import re
from collections import Counter, defaultdict

WORD_RE = re.compile(r"\w+(?:['’-]\w+)*")
token_words = defaultdict(Counter)  # token id -> Counter of source words
for sent, _ in dataset:
    if not sent:
        continue
    enc = tokenizer(sent, add_special_tokens=False, return_offsets_mapping=True)
    words = [(m.start(), m.end(), m.group()) for m in WORD_RE.finditer(sent)]
    for tid, (s, e) in zip(enc["input_ids"], enc["offset_mapping"]):
        for ws, we, w in words:
            if ws < e and s < we:  # token's character span overlaps this word
                token_words[tid][w] += 1

tables = {}
for feat in F_C:
    mags = Y[feat].abs()
    nz_idx = torch.nonzero(mags > 0).flatten()
    nz_idx = nz_idx[torch.argsort(mags[nz_idx], descending=True)].tolist()
    ids = [vstar_ids[j] for j in nz_idx]
    tables[f"feature {feat}"] = pd.DataFrame({
        "token": tokenizer.convert_ids_to_tokens(ids),
        "label": [LABEL_TAG[codes[j]] for j in nz_idx],
        "|Y|": [round(mags[j].item(), 2) for j in nz_idx],
        "words": [[w for w, _ in token_words[t].most_common()] for t in ids],
    })

wide_split = pd.concat(tables, axis=1).fillna("")
wide_split.index = range(1, len(wide_split) + 1)
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", None):
    display(wide_split)


feature 85              \
          token label   |Y|   
1       ffindor     C  0.35   
2        therin     C  0.29   
3     umbledore     C  0.25   
4   ▁Dumbledore     C  0.22   
5      ▁Weasley     C  0.21   
6     ▁Hogwarts     C  0.20   
7       ▁Malfoy     C  0.20   
8      ▁Rowling     C  0.20   
9       ▁Potter     C  0.20   
10    ▁Hermione     C  0.19   
11       demort     C  0.19   
12     Hermione     C  0.19   
13   ▁Voldemort     C  0.19   
14       ▁Harry     C  0.17   
15       Potter     C  0.17   
16       ▁Snape     C  0.17   
17        Harry     C  0.16   
18     ▁Severus     C  0.13   
19       ▁Draco     C  0.13   
20          Hog     C  0.13   
21         ▁Gry     C  0.12   
22       ▁Ginny     C  0.11   
23         ▁Hog     C  0.11   
24         otts     C  0.10   
25        warts     C  0.10   
26       ▁Lupin     C  0.08   
27       ▁Remus     C  0.08   
28     ▁Granger     C  0.08   
29         ▁Sly     C  0.08   
30          ▁JK     C  0.08   
31      ▁Sirius     C  0.07   
32        ▁wand     C  0.07   
33         ▁McG     C  0.07   
34         ▁Mug     B  0.07   
35         ▁Ron     C  0.07   
36         blet     C  0.07   
37    ▁Sherlock     C  0.07   
38         ▁Hag     C  0.06   
39          idd     C  0.06   
40       ▁Beaux     B  0.06   
41     ▁potions     C  0.06   
42          Row     C  0.06   
43      ▁Riddle     C  0.06   
44        ▁Grin     C  0.06   
45        ▁Quir     C  0.06   
46      ▁Holmes     C  0.06   
47      ▁Merlin     C  0.06   
48           HP     N  0.06   

                                                               
                                                        words  
1                                                [Gryffindor]  
2                                               [Slytherin's]  
3                                  [Dumbledore's, Dumbledore]  
4                                  [Dumbledore, Dumbledore's]  
5                                                   [Weasley]  
6                                                  [Hogwarts]  
7                                                    [Malfoy]  
8                                        [Rowling, Rowling's]  
9   [Potter, Potter's, Potter-themed, Pottermore, Potter's's]  
10                                     [Hermione, Hermione's]  
11                                                [Voldemort]  
12                                                 [Hermione]  
13                                   [Voldemort, Voldemort's]  
14                                           [Harry, Harry's]  
15                                               [Pottermore]  
16                                           [Snape, Snape's]  
17                                           [Harry, Harry's]  
18                                                  [Severus]  
19                                                    [Draco]  
20                                                 [Hogwarts]  
21                                               [Gryffindor]  
22                                                    [Ginny]  
23                                                [Hogsmeade]  
24                                                [Gringotts]  
25                                                 [Hogwarts]  
26                                                    [Lupin]  
27                                                    [Remus]  
28                                                  [Granger]  
29                                              [Slytherin's]  
30                                                     [JK's]  
31                                                   [Sirius]  
32                                                     [wand]  
33                                               [McGonagall]  
34                                   [Muggles, Muggle, Muggs]  
35                                               [Ron, Ron's]  
36                                                   [Goblet]  
37                    

In [ ]:
TOP_N = 30
E_ref = E_vstar_X.float().cpu()  # X-space rows = transform(E_vstar): the space Z lives in
token_strs = tokenizer.convert_ids_to_tokens(vstar_ids)

group_masks = {
    "C": codes == 1,
    "C / B": (codes == 1) | (codes == 2),
    "C / B / N": np.ones_like(codes, dtype=bool),
}

for feat in F_C:
    y = Y[feat]
    # z's sign is arbitrary (Y can absorb it); orient it so its support tokens mostly have positive Y.
    sign = 1.0 if y[y != 0].sum() >= 0 else -1.0
    z = sign * Z[:, feat]
    dots = E_ref @ z
    cos = dots / (E_ref.norm(dim=1) * z.norm())

    df = pd.DataFrame({
        "token": token_strs,
        "label": [LABEL_TAG[c] for c in codes],
        "dot": dots.numpy(),
        "cos": cos.numpy(),
        "Y": (sign * y).numpy(),
        "in_support": (y != 0).numpy(),
    })

    print(f"=== feature {feat}   ||z|| = {z.norm():.3f}   sign flipped: {sign < 0} ===")
    for name, mask in group_masks.items():
        top = df[mask].sort_values("dot", ascending=False).head(TOP_N).reset_index(drop=True)
        top.index += 1
        print(f"\n-- {name}: {int(mask.sum())} tokens, top {TOP_N} by <e_t, z> --")
        display(top)

=== feature 85   ||z|| = 187.930   sign flipped: True ===



-- C: 1870 tokens, top 30 by <e_t, z> --


,token,label,dot,cos,Y,in_support
1,ffindor,C,10874.533203,0.504992,0.347119,True
2,therin,C,9494.877930,0.506701,0.287267,True
3,umbledore,C,8824.896484,0.556008,0.253115,True
4,▁Dumbledore,C,7769.307129,0.666268,0.220213,True
5,▁Hogwarts,C,7755.325195,0.673038,0.204527,True
6,▁Weasley,C,7206.792480,0.581711,0.213825,True
7,▁Malfoy,C,6992.492676,0.601465,0.201548,True
8,▁Voldemort,C,6954.417969,0.623283,0.187210,True
9,▁Hermione,C,6933.039062,0.651407,0.192658,True
10,demort,C,6712.996582,0.431761,0.191899,True



-- C / B: 2551 tokens, top 30 by <e_t, z> --


,token,label,dot,cos,Y,in_support
1,ffindor,C,10874.533203,0.504992,0.347119,True
2,therin,C,9494.877930,0.506701,0.287267,True
3,umbledore,C,8824.896484,0.556008,0.253115,True
4,▁Dumbledore,C,7769.307129,0.666268,0.220213,True
5,▁Hogwarts,C,7755.325195,0.673038,0.204527,True
6,▁Weasley,C,7206.792480,0.581711,0.213825,True
7,▁Malfoy,C,6992.492676,0.601465,0.201548,True
8,▁Voldemort,C,6954.417969,0.623283,0.187210,True
9,▁Hermione,C,6933.039062,0.651407,0.192658,True
10,demort,C,6712.996582,0.431761,0.191899,True



-- C / B / N: 4780 tokens, top 30 by <e_t, z> --


,token,label,dot,cos,Y,in_support
1,ffindor,C,10874.533203,0.504992,0.347119,True
2,therin,C,9494.877930,0.506701,0.287267,True
3,umbledore,C,8824.896484,0.556008,0.253115,True
4,▁Dumbledore,C,7769.307129,0.666268,0.220213,True
5,▁Hogwarts,C,7755.325195,0.673038,0.204527,True
6,▁Weasley,C,7206.792480,0.581711,0.213825,True
7,▁Malfoy,C,6992.492676,0.601465,0.201548,True
8,▁Voldemort,C,6954.417969,0.623283,0.187210,True
9,▁Hermione,C,6933.039062,0.651407,0.192658,True
10,demort,C,6712.996582,0.431761,0.191899,True


In [ ]:
E_unit = E_ref / E_ref.norm(dim=1, keepdim=True)

for feat in F_C:
    y = Y[feat]
    sign = 1.0 if y[y != 0].sum() >= 0 else -1.0
    z = sign * Z[:, feat]
    dots_unit = E_unit @ z

    df_unit = pd.DataFrame({
        "token": token_strs,
        "label": [LABEL_TAG[c] for c in codes],
        "dot_unit": dots_unit.numpy(),
        "||e_t||": E_ref.norm(dim=1).numpy(),
        "Y": (sign * y).numpy(),
        "in_support": (y != 0).numpy(),
    })

    print(f"=== feature {feat}   ||z|| = {z.norm():.3f}   sign flipped: {sign < 0} ===")
    for name, mask in group_masks.items():
        top = df_unit[mask].sort_values("dot_unit", ascending=False).head(TOP_N).reset_index(drop=True)
        top.index += 1
        print(f"\n-- {name}: {int(mask.sum())} tokens, top {TOP_N} by <e_t / ||e_t||, z> --")
        display(top)

=== feature 85   ||z|| = 187.930   sign flipped: True ===

-- C: 1870 tokens, top 30 by <e_t / ||e_t||, z> --


,token,label,dot_unit,||e_t||,Y,in_support
1,▁Hogwarts,C,126.484138,61.314606,0.204527,True
2,▁Dumbledore,C,125.211861,62.049290,0.220213,True
3,▁Hermione,C,122.419075,56.633656,0.192658,True
4,▁Voldemort,C,117.133682,59.371643,0.187210,True
5,▁Malfoy,C,113.033562,61.862099,0.201548,True
6,▁Weasley,C,109.321098,65.923157,0.213825,True
7,umbledore,C,104.490776,84.456223,0.253115,True
8,▁Harry,C,102.402740,59.297260,0.174415,True
9,▁Potter,C,102.360382,64.879997,0.195160,True
10,Hermione,C,101.910507,64.814766,0.188506,True



-- C / B: 2551 tokens, top 30 by <e_t / ||e_t||, z> --


,token,label,dot_unit,||e_t||,Y,in_support
1,▁Hogwarts,C,126.484138,61.314606,0.204527,True
2,▁Dumbledore,C,125.211861,62.049290,0.220213,True
3,▁Hermione,C,122.419075,56.633656,0.192658,True
4,▁Voldemort,C,117.133682,59.371643,0.187210,True
5,▁Malfoy,C,113.033562,61.862099,0.201548,True
6,▁Weasley,C,109.321098,65.923157,0.213825,True
7,umbledore,C,104.490776,84.456223,0.253115,True
8,▁Harry,C,102.402740,59.297260,0.174415,True
9,▁Potter,C,102.360382,64.879997,0.195160,True
10,Hermione,C,101.910507,64.814766,0.188506,True



-- C / B / N: 4780 tokens, top 30 by <e_t / ||e_t||, z> --


,token,label,dot_unit,||e_t||,Y,in_support
1,▁Hogwarts,C,126.484138,61.314606,0.204527,True
2,▁Dumbledore,C,125.211861,62.049290,0.220213,True
3,▁Hermione,C,122.419075,56.633656,0.192658,True
4,▁Voldemort,C,117.133682,59.371643,0.187210,True
5,▁Malfoy,C,113.033562,61.862099,0.201548,True
6,▁Weasley,C,109.321098,65.923157,0.213825,True
7,umbledore,C,104.490776,84.456223,0.253115,True
8,▁Harry,C,102.402740,59.297260,0.174415,True
9,▁Potter,C,102.360382,64.879997,0.195160,True
10,Hermione,C,101.910507,64.814766,0.188506,True


In [ ]:
TOP_K_COUNT = 100

rows = []
for feat in F_C:
    y = Y[feat]
    sign = 1.0 if y[y != 0].sum() >= 0 else -1.0
    z = sign * Z[:, feat]
    rankings = {
        "<e_t, z>": E_ref @ z,
        "<e_t / ||e_t||, z>": E_unit @ z,
    }
    for name, scores in rankings.items():
        top = torch.argsort(scores, descending=True)[:TOP_K_COUNT].numpy()
        rows.append({
            "feature": feat,
            "ranking": name,
            "C": int((codes[top] == 1).sum()),
            "B": int((codes[top] == 2).sum()),
            "N": int((codes[top] == 0).sum()),
            "C in support": int(((codes[top] == 1) & (y[top] != 0).numpy()).sum()),
        })

print(f"Label counts among the top {TOP_K_COUNT} of all {len(vstar_ids)} tokens in V* "
      f"(base rate: C={(codes == 1).mean():.1%}, B={(codes == 2).mean():.1%}, N={(codes == 0).mean():.1%})")
pd.DataFrame(rows)

Label counts among the top 100 of all 4780 tokens in V* (base rate: C=39.1%, B=14.2%, N=46.6%)


,feature,ranking,C,B,N,C in support
0,85,"<e_t, z>",93,6,1,45
1,85,"<e_t / ||e_t||, z>",90,4,6,45


### 3.2.g Extended token set: every sentence of the concept's Wikipedia page, $Z$ fixed
$S_C^{ext} = S_C \cup$ all sentences of the article, $S_N^{ext} = S_N \cup$ as many neutral sentences. The new rows go through the **same** transform $T$ fitted in 3.1.c′ (nothing re-estimated), then only a sparse $Y_{ext}$ is fitted against the frozen X-space $Z$. Groups: C / B / N = label in the original $V^*$ (3.2.a), C\* / B\* / N\* = label on the extended sentences; a token gets the first group it matches in the order C, C\*, B, B\*, N, N\*.

In [ ]:
# ---- Extended E*: all sentences of the concept's Wikipedia page, Z from 3.1.d fixed, only a sparse Y_ext refit ----
# S_C^ext = S_C + every sentence of the live English Wikipedia article.
# S_N^ext = S_N + neutral sentences (one per page of the local wikimedia/wikipedia sample), added one at a time until
# the new neutral-only tokens outnumber the new concept-only ones (N* > C*; B* is counted but not compared), so the
# neutral side does not grow out of proportion to what the page adds. The candidate sentences are cached next to
# RUN_DIR, so re-runs use the same page revision and sample.
# Rows go to X-space with the transform T of 3.1.c' exactly as fitted there (same mu_B, W, S_eta: nothing re-estimated),
# then Y_ext = WTA_s((Z^T Z + lambda I)^{-1} Z^T X_ext^T): the Y-step of 3.1.d with Z frozen and the same lambda and s.
import random
import sys
from collections import Counter

from ember.utils import wta_cols

sys.path.insert(0, "experiments/resample_sentences")
from fetch_en_sentences import neutral_sentences, page_paragraphs, split_sentences

EXT_PATH = RUN_DIR.parent / "ext_sentences.json"   # shared by every whitening config of this concept and model
EXT_GROUPS = ["C", "C*", "B", "B*", "N", "N*"]

# 1) candidate sentences: the whole page, and a pool of as many neutral sentences
ext = load_json(EXT_PATH, None)
if ext is None:
    canon, revid, paras = page_paragraphs(CONCEPT)
    page_sents = split_sentences(paras)
    ext = {"title": canon, "revid": revid, "concept": page_sents,
           "neutral": [r["sentence"] for r in neutral_sentences(
               Path("experiments/resample_sentences/data/wiki_dump_pages.jsonl"), len(page_sents),
               random.Random(SEED), exclude=set(page_sents))]}
    save_json(EXT_PATH, ext)

special_ext = set(tokenizer.all_special_ids)
vstar_set = set(vstar_ids)
orig_code = dict(zip(vstar_ids, codes.tolist()))


def sent_tokens(sent: str) -> set:
    return {int(t) for t in tokenizer(sent, add_special_tokens=False)["input_ids"]} - special_ext


def ext_group(t: int, in_c: bool, in_n: bool) -> str:
    """First match in the order C, C*, B, B*, N, N* (C/B/N: label in V*, starred: label on the extended sentences)."""
    o = orig_code.get(t)
    c_ext = 2 if in_c and in_n else 1 if in_c else 0
    for g, hit in [("C", o == 1), ("C*", c_ext == 1), ("B", o == 2), ("B*", c_ext == 2), ("N", o == 0), ("N*", c_ext == 0)]:
        if hit:
            return g


def group_counts() -> Counter:
    return Counter(ext_group(t, t in c_ids, t in n_ids) for t in c_ids | n_ids | vstar_set)


# 2a) concept side first: the original sample + every page sentence
sampled_concept = list(dict.fromkeys(s for s, lbl in dataset if lbl == CONCEPT))
ext_concept = list(dict.fromkeys(sampled_concept + ext["concept"]))
c_ids = set().union(*map(sent_tokens, ext_concept))

# 2b) neutral side: the original sample, then pool sentences one at a time until N* > C*
sampled_neutral = list(dict.fromkeys(s for s, lbl in dataset if lbl != CONCEPT))
ext_neutral = list(sampled_neutral)
n_ids = set().union(*map(sent_tokens, ext_neutral))
counts = group_counts()
trace = [{"added": 0, "C*": counts["C*"], "B*": counts["B*"], "N*": counts["N*"]}]
seen = set(ext_neutral)
for sent in ext["neutral"]:
    if counts["N*"] > counts["C*"]:
        break
    if sent in seen:
        continue
    seen.add(sent)
    ext_neutral.append(sent)
    n_ids |= sent_tokens(sent)
    counts = group_counts()
    trace.append({"added": len(trace), "C*": counts["C*"], "B*": counts["B*"], "N*": counts["N*"]})
if counts["N*"] <= counts["C*"]:
    print(f"WARNING: the pool of {len(ext['neutral'])} neutral sentences ran out before N* passed C*")
trace = pd.DataFrame(trace).set_index("added")

n_page_new = len(ext_concept) - len(sampled_concept)   # page sentences not already in the sample
n_added = len(ext_neutral) - len(sampled_neutral)
print(f"'{ext['title']}' rev {ext['revid']} ({EXT_PATH})\n"
      f"S_C^ext = {len(ext_concept)} sentences: {len(sampled_concept)} sampled + {n_page_new} new from the page "
      f"({len(ext['concept'])} page sentences, {len(ext['concept']) - n_page_new} of them already sampled)\n"
      f"S_N^ext = {len(ext_neutral)} sentences: {len(sampled_neutral)} sampled + {n_added} added "
      f"from a pool of {len(ext['neutral'])}\n"
      f"stop: C* = {counts['C*']}  B* = {counts['B*']}  N* = {counts['N*']}")
# display(trace.iloc[np.unique(np.linspace(0, len(trace) - 1, 11).astype(int))].T)   # ~10 evenly spaced steps

# 2c) the extended token set (a superset of V*) and its groups
vstar_ext = sorted(c_ids | n_ids | vstar_set)
codes_ext = np.array([2 if t in c_ids and t in n_ids else 1 if t in c_ids else 0 for t in vstar_ext])
groups_ext = np.array([ext_group(t, t in c_ids, t in n_ids) for t in vstar_ext])
print(f"|V*_ext| = {len(vstar_ext)}  (|V*| = {len(vstar_ids)}, {len(vstar_ext) - len(vstar_ids)} new)")
display(pd.Series(groups_ext).value_counts().reindex(EXT_GROUPS).fillna(0).astype(int).rename("tokens").to_frame().T)

# 3) X_ext = T(E_{V*_ext}): the 3.1.c' transform, unchanged
ids_ext = torch.tensor(vstar_ext)
if "E" in globals():
    E_ext = E.index_select(0, ids_ext.to(E.device)).detach().float().cpu()
else:   # after 3.3.a the TL matrix is gone; HF rows x EMB_SCALE are the same space (checked in 3.3.a)
    E_ext = W_orig.index_select(0, ids_ext.to(W_orig.device)).detach().float().cpu() * EMB_SCALE
X_ext = transform(E_ext)                                                   # (|V*_ext|, d), X-space
pos_ext = {t: j for j, t in enumerate(vstar_ext)}
in_vstar = [pos_ext[t] for t in vstar_ids]
print(f"check: max |X_ext[V*] - E_vstar_X| = {(X_ext[in_vstar] - E_vstar_X.float().cpu()).abs().max():.2e}")

# 4) Y-step with Z frozen (same ops as SparseMatrixFactorization.fit, incl. ridge and WTA)
A_ext = X_ext.T.contiguous().to(DEVICE)                                    # (d, |V*_ext|)
Z_dev = Z.to(DEVICE)
with Timer() as t_ext:
    inv_Z = torch.linalg.inv(Z_dev.T @ Z_dev + torch.eye(Z_dev.shape[1], device=DEVICE) * ridge_used)
    G_ext = A_ext.T @ Z_dev @ inv_Z                                        # (|V*_ext|, k), the G_ layout
    wta_cols(G_ext, pct_keep=SPARSITY)
Y_ext = G_ext.T.float().cpu()                                              # (k, |V*_ext|)
A_ext_cpu = A_ext.cpu()
ext_err = (torch.norm(A_ext_cpu - Z @ Y_ext) / torch.norm(A_ext_cpu)).item()
print(f"fixed-Z fit: {t_ext['elapsed']:.3f}s, WTA keeps {math.ceil(SPARSITY * len(vstar_ext))} tokens per feature")
print(f"relative error: on X_ext (fixed Z) = {ext_err:.4f}   3.1.d joint fit on A = {rel_err:.4f}")
del A_ext, Z_dev, G_ext, A_ext_cpu, E_ext


'Harry Potter' rev 1379437311 (runs/whiten/gemma/Harry_Potter/ext_sentences.json)
S_C^ext = 444 sentences: 300 sampled + 144 new from the page (403 page sentences, 259 of them already sampled)
S_N^ext = 368 sentences: 280 sampled + 88 added from a pool of 403
stop: C* = 515  B* = 108  N* = 523
|V*_ext| = 5835  (|V*| = 4780, 1055 new)


,C,C*,B,B*,N,N*
tokens,1870,515,681,108,2138,523


check: max |X_ext[V*] - E_vstar_X| = 0.00e+00
fixed-Z fit: 0.005s, WTA keeps 59 tokens per feature
relative error: on X_ext (fixed Z) = 0.8714   3.1.d joint fit on A = 0.8578


In [ ]:
# ---- Rate every token of V*_ext on each concept feature, then list the feature's Y_ext support (X-space, where Z lives) ----
#   <x_t, z~>            inner product of the transformed token row with the feature direction
#   <x_t / ||x_t||, z~>  the same with the token row normalized
#   words                whole words of S_C^ext and S_N^ext the token was cut from (most frequent first)
# z~'s sign is arbitrary (Y absorbs it): oriented as in 3.2.f, so its 3.1 support tokens mostly have positive Y.
# Support table: tokens with Y_ext[feat, t] != 0, by decreasing |Y_ext|; group = first match in C, C*, B, B*, N, N* (3.2.g);
# in V*: the token is in the original token set;  in original support: Y[feat, t] != 0 in the 3.1 fit.
import re
from collections import Counter, defaultdict

SHOW_ROWS = 100   # rows shown per ranking; None shows all |V*_ext| tokens (the full tables are in ext_rating)

WORD_RE = re.compile(r"\w+(?:['’-]\w+)*")
token_words_ext = defaultdict(Counter)  # token id -> Counter of source words
for sent in ext_concept + ext_neutral:
    enc = tokenizer(sent, add_special_tokens=False, return_offsets_mapping=True)
    words = [(m.start(), m.end(), m.group()) for m in WORD_RE.finditer(sent)]
    for tid, (s, e) in zip(enc["input_ids"], enc["offset_mapping"]):
        for ws, we, w in words:
            if ws < e and s < we:  # token's character span overlaps this word
                token_words_ext[tid][w] += 1
words_ext = [[w for w, _ in token_words_ext[t].most_common()] for t in vstar_ext]

ext_strs = tokenizer.convert_ids_to_tokens(vstar_ext)
x_norms = X_ext.norm(dim=1)
vstar_set = set(vstar_ids)
ext_rating, ext_support = {}, {}
for feat in F_C:
    y = Y[feat]
    sign = 1.0 if y[y != 0].sum() >= 0 else -1.0
    z = sign * Z[:, feat]
    dots = X_ext @ z
    orig_sup = {vstar_ids[j] for j in torch.nonzero(y).flatten().tolist()}
    ext_sup = (Y_ext[feat] != 0).numpy()

    # 1) every token of V*_ext, ranked by the two inner products
    # support: first match of (original: in this direction's 3.1 support, extended: in its Y_ext support, none)
    df = pd.DataFrame({
        "token": ext_strs,
        "group": groups_ext,
        "words": words_ext,
        "support": ["original" if t in orig_sup else "extended" if e else "none" for t, e in zip(vstar_ext, ext_sup)],
        "<x_t, z>": dots.numpy(),
        "<x_t / ||x_t||, z>": (dots / x_norms).numpy(),
        "||x_t||": x_norms.numpy(),
        "Y_ext": (sign * Y_ext[feat]).numpy(),
        "in Y_ext support": (Y_ext[feat] != 0).numpy(),
    })
    ext_rating[feat] = df

    print(f"{'=' * 100}\n=== feature {feat}   ||z|| = {z.norm():.3f}   sign flipped: {sign < 0}   "
          f"|V*_ext| = {len(df)}   Y_ext support = {int(df['in Y_ext support'].sum())} ===\n{'=' * 100}")
    by_group = df.groupby("group")[["<x_t, z>", "<x_t / ||x_t||, z>"]].agg(["mean", "median", "max"])
    by_group.insert(0, "tokens", df["group"].value_counts())
    display(by_group.reindex(EXT_GROUPS).style.format("{:.3f}", subset=by_group.columns[1:]))
    for col in ["<x_t, z>", "<x_t / ||x_t||, z>"]:
        ranked = df.sort_values(col, ascending=False, ignore_index=True)
        ranked.index += 1
        print(f"\n-- all {len(df)} tokens of V*_ext by {col} (top {SHOW_ROWS or len(df)} shown) --")
        with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
            display(ranked if SHOW_ROWS is None else ranked.head(SHOW_ROWS))

    # 2) the feature's Y_ext support, by decreasing |Y_ext|
    sup = torch.nonzero(Y_ext[feat]).flatten()
    sup = sup[torch.argsort(Y_ext[feat, sup].abs(), descending=True)].tolist()
    sup_df = pd.DataFrame({
        "token": [ext_strs[j] for j in sup],
        "group": groups_ext[sup],
        "words": [words_ext[j] for j in sup],
        "Y_ext": (sign * Y_ext[feat, sup]).numpy(),
        "in V*": [vstar_ext[j] in vstar_set for j in sup],
        "in original support": [vstar_ext[j] in orig_sup for j in sup],
        "<x_t, z>": dots[sup].numpy(),
        "<x_t / ||x_t||, z>": (dots[sup] / x_norms[sup]).numpy(),
    }, index=pd.RangeIndex(1, len(sup) + 1, name="rank"))
    ext_support[feat] = sup_df

    lost = sorted(orig_sup - {vstar_ext[j] for j in sup})
    counts = "  ".join(f"{g}: {int((sup_df['group'] == g).sum())}" for g in EXT_GROUPS)
    print(f"\n-- Y_ext support: {len(sup_df)} tokens [{counts}] | in V*: {int(sup_df['in V*'].sum())} | "
          f"in original support: {int(sup_df['in original support'].sum())}/{len(orig_sup)} --")
    if lost:
        print(f"original support tokens no longer in the support ({len(lost)}): {tokenizer.convert_ids_to_tokens(lost)}")
    with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
        display(sup_df.style.format({"Y_ext": "{:+.4f}", "<x_t, z>": "{:.2f}", "<x_t / ||x_t||, z>": "{:.3f}"}))


=== feature 85   ||z|| = 187.930   sign flipped: True   |V*_ext| = 5835   Y_ext support = 59 ===



-- all 5835 tokens of V*_ext by <x_t, z> (top 100 shown) --


,token,group,words,support,"<x_t, z>","<x_t / ||x_t||, z>",||x_t||,Y_ext,in Y_ext support
1,ffindor,C,[Gryffindor],original,10874.533203,94.903214,114.585510,0.347119,True
2,therin,C,[Slytherin's],original,9494.877930,95.224388,99.710571,0.287267,True
3,umbledore,C,"[Dumbledore's, Dumbledore]",original,8824.896484,104.490776,84.456223,0.253115,True
4,▁Dumbledore,C,"[Dumbledore, Dumbledore's]",original,7769.307129,125.211861,62.049290,0.220213,True
5,▁Hogwarts,C,[Hogwarts],original,7755.325195,126.484138,61.314606,0.204527,True
6,▁Weasley,C,[Weasley],original,7206.792480,109.321106,65.923157,0.213825,True
7,▁Malfoy,C,[Malfoy],original,6992.492676,113.033554,61.862099,0.201548,True
8,▁Voldemort,C,"[Voldemort, Voldemort's]",original,6954.417969,117.133659,59.371643,0.187210,True
9,▁Hermione,C,"[Hermione, Hermione's]",original,6933.039062,122.419060,56.633656,0.192658,True
10,demort,C,[Voldemort],original,6712.996582,81.140945,82.732544,0.191899,True



-- all 5835 tokens of V*_ext by <x_t / ||x_t||, z> (top 100 shown) --


,token,group,words,support,"<x_t, z>","<x_t / ||x_t||, z>",||x_t||,Y_ext,in Y_ext support
1,▁Hogwarts,C,[Hogwarts],original,7755.325195,126.484138,61.314606,0.204527,True
2,▁Dumbledore,C,"[Dumbledore, Dumbledore's]",original,7769.307129,125.211861,62.049290,0.220213,True
3,▁Hermione,C,"[Hermione, Hermione's]",original,6933.039062,122.419060,56.633656,0.192658,True
4,▁Voldemort,C,"[Voldemort, Voldemort's]",original,6954.417969,117.133659,59.371643,0.187210,True
5,▁Malfoy,C,[Malfoy],original,6992.492676,113.033554,61.862099,0.201548,True
6,▁Weasley,C,[Weasley],original,7206.792480,109.321106,65.923157,0.213825,True
7,umbledore,C,"[Dumbledore's, Dumbledore]",original,8824.896484,104.490776,84.456223,0.253115,True
8,▁Harry,C,"[Harry, Harry's, Harryplax]",original,6072.201660,102.402733,59.297260,0.174415,True
9,▁Potter,C,"[Potter, Potter's, Potter-themed, Pottermore, Potter's's]",original,6641.141602,102.360390,64.879997,0.195160,True
10,Hermione,C,[Hermione],original,6605.305664,101.910507,64.814766,0.188506,True



-- Y_ext support: 59 tokens [C: 51  C*: 4  B: 2  B*: 0  N: 1  N*: 1] | in V*: 54 | in original support: 48/48 --


,token,group,words,Y_ext,in V*,in original support,,
rank,,,,,,,,
1,ffindor,C,['Gryffindor'],+0.3471,True,True,10874.53,94.903
2,therin,C,"[""Slytherin's""]",+0.2873,True,True,9494.88,95.224
3,umbledore,C,"[""Dumbledore's"", 'Dumbledore']",+0.2531,True,True,8824.90,104.491
4,▁Dumbledore,C,"['Dumbledore', ""Dumbledore's""]",+0.2202,True,True,7769.31,125.212
5,▁Weasley,C,['Weasley'],+0.2138,True,True,7206.79,109.321
6,▁Hogwarts,C,['Hogwarts'],+0.2045,True,True,7755.33,126.484
7,▁Malfoy,C,['Malfoy'],+0.2015,True,True,6992.49,113.034
8,▁Rowling,C,"['Rowling', ""Rowling's""]",+0.1976,True,True,6677.82,98.176
9,▁Potter,C,"['Potter', ""Potter's"", 'Potter-themed', 'Pottermore', ""Potter's's""]",+0.1952,True,True,6641.14,102.360


In [ ]:
# ---- Dense refit: fix Z, solve Y' = argmin ||E*^T - Z Y'||_F (no sparsity) ----
TOP_DIRS = 5
HP_FEATS = list(F_C)          # the Harry Potter concept column(s) of Z; override e.g. [37] if you want one
HP_MARK = "★★★ HP ★★★"

A_ref = E_vstar_X.double().cpu().T                                                # (d, |V*|), X-space
Z64 = Z.double()                                                                    # (d, k)
Y_dense = torch.linalg.lstsq(Z64, A_ref).solution.float()                           # (k, |V*|)

dense_err = (torch.norm(A_ref - Z64 @ Y_dense.double()) / torch.norm(A_ref)).item()
sparse_err = (torch.norm(A_ref - Z64 @ Y.double()) / torch.norm(A_ref)).item()
print(f"relative error: sparse Y = {sparse_err:.4f}   dense Y' = {dense_err:.4f}")
print(f"HP feature(s): {HP_FEATS}   (||z|| = {[round(Z[:, f].norm().item(), 3) for f in HP_FEATS]})\n")

for hp in HP_FEATS:
    mags = Y[hp].abs()
    support = torch.nonzero(mags > 0).flatten()
    support = support[torch.argsort(mags[support], descending=True)].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in support])

    print(f"{'=' * 100}\n=== HP feature {hp}: {len(support)} tokens with nonzero Y, by decreasing |Y| ===\n{'=' * 100}")
    for rank_t, (j, tok) in enumerate(zip(support, toks), 1):
        y_dense_t = Y_dense[:, j]
        order = torch.argsort(y_dense_t.abs(), descending=True).tolist()
        top = order[:TOP_DIRS]

        print(f"\n{rank_t:3d}. {tok!r} [{LABEL_TAG[codes[j]]}]   sparse Y[{hp}] = {Y[hp, j]:+.3f}")
        for r, f in enumerate(top, 1):
            mark = f"   <-- {HP_MARK}" if f in HP_FEATS else ""
            print(f"       #{r}  z_{f:<4d} Y' = {y_dense_t[f]:+.3f}{mark}")
        for f in HP_FEATS:
            if f not in top:
                print(f"       ..  z_{f:<4d} Y' = {y_dense_t[f]:+.3f}   <-- {HP_MARK} (rank {order.index(f) + 1}/{len(order)})")

relative error: sparse Y = 0.8578   dense Y' = 0.8118
HP feature(s): [85]   (||z|| = [187.93])

=== HP feature 85: 48 tokens with nonzero Y, by decreasing |Y| ===

  1. 'ffindor' [C]   sparse Y[85] = -0.347
       #1  z_85   Y' = -0.347   <-- ★★★ HP ★★★
       #2  z_88   Y' = -0.163
       #3  z_67   Y' = +0.142
       #4  z_83   Y' = +0.112
       #5  z_46   Y' = -0.087

  2. 'therin' [C]   sparse Y[85] = -0.287
       #1  z_85   Y' = -0.287   <-- ★★★ HP ★★★
       #2  z_88   Y' = -0.106
       #3  z_67   Y' = +0.081
       #4  z_69   Y' = +0.081
       #5  z_91   Y' = +0.067

  3. 'umbledore' [C]   sparse Y[85] = -0.253
       #1  z_85   Y' = -0.253   <-- ★★★ HP ★★★
       #2  z_83   Y' = -0.042
       #3  z_29   Y' = -0.037
       #4  z_84   Y' = -0.031
       #5  z_49   Y' = -0.030

  4. '▁Dumbledore' [C]   sparse Y[85] = -0.220
       #1  z_85   Y' = -0.220   <-- ★★★ HP ★★★
       #2  z_96   Y' = -0.020
       #3  z_54   Y' = -0.019
       #4  z_29   Y' = -0.017
       #5  z_31   

In [ ]:
# ---- Tokens where the HP direction dominates Y': top-1 is HP and |Y'_1| >= RATIO * |Y'_2| ----
RATIO = 2.0

for hp in HP_FEATS:
    mags = Y[hp].abs()
    support = torch.nonzero(mags > 0).flatten()
    support = support[torch.argsort(mags[support], descending=True)].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in support])

    kept = []
    for rank_t, (j, tok) in enumerate(zip(support, toks), 1):
        y_dense_t = Y_dense[:, j]
        order = torch.argsort(y_dense_t.abs(), descending=True).tolist()
        first, second = y_dense_t[order[0]].abs().item(), y_dense_t[order[1]].abs().item()
        if order[0] == hp and first >= RATIO * second:
            kept.append((rank_t, j, tok, order, first / second))

    print(f"{'=' * 100}\n=== HP feature {hp}: {len(kept)} / {len(support)} tokens where z_{hp} is top-1 "
          f"and |Y'| >= {RATIO:g}x the runner-up ===\n{'=' * 100}")
    for rank_t, j, tok, order, ratio in kept:
        y_dense_t = Y_dense[:, j]
        print(f"\n{rank_t:3d}. {tok!r} [{LABEL_TAG[codes[j]]}]   sparse Y[{hp}] = {Y[hp, j]:+.3f}   "
              f"top-1/top-2 = {ratio:.2f}x")
        for r, f in enumerate(order[:TOP_DIRS], 1):
            mark = f"   <-- {HP_MARK}" if f in HP_FEATS else ""
            print(f"       #{r}  z_{f:<4d} Y' = {y_dense_t[f]:+.3f}{mark}")

=== HP feature 85: 41 / 48 tokens where z_85 is top-1 and |Y'| >= 2x the runner-up ===

  1. 'ffindor' [C]   sparse Y[85] = -0.347   top-1/top-2 = 2.13x
       #1  z_85   Y' = -0.347   <-- ★★★ HP ★★★
       #2  z_88   Y' = -0.163
       #3  z_67   Y' = +0.142
       #4  z_83   Y' = +0.112
       #5  z_46   Y' = -0.087

  2. 'therin' [C]   sparse Y[85] = -0.287   top-1/top-2 = 2.71x
       #1  z_85   Y' = -0.287   <-- ★★★ HP ★★★
       #2  z_88   Y' = -0.106
       #3  z_67   Y' = +0.081
       #4  z_69   Y' = +0.081
       #5  z_91   Y' = +0.067

  3. 'umbledore' [C]   sparse Y[85] = -0.253   top-1/top-2 = 6.08x
       #1  z_85   Y' = -0.253   <-- ★★★ HP ★★★
       #2  z_83   Y' = -0.042
       #3  z_29   Y' = -0.037
       #4  z_84   Y' = -0.031
       #5  z_49   Y' = -0.030

  4. '▁Dumbledore' [C]   sparse Y[85] = -0.220   top-1/top-2 = 11.07x
       #1  z_85   Y' = -0.220   <-- ★★★ HP ★★★
       #2  z_96   Y' = -0.020
       #3  z_54   Y' = -0.019
       #4  z_29   Y' = -0.017
     

In [ ]:
# ---- Tokens skipped by the previous cell: HP is not top-1, or top-1 but < RATIO x the runner-up ----
for hp in HP_FEATS:
    mags = Y[hp].abs()
    support = torch.nonzero(mags > 0).flatten()
    support = support[torch.argsort(mags[support], descending=True)].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in support])

    skipped = []
    for rank_t, (j, tok) in enumerate(zip(support, toks), 1):
        y_dense_t = Y_dense[:, j]
        order = torch.argsort(y_dense_t.abs(), descending=True).tolist()
        first, second = y_dense_t[order[0]].abs().item(), y_dense_t[order[1]].abs().item()
        if order[0] != hp:
            skipped.append((rank_t, j, tok, order, f"HP is not top-1 (rank {order.index(hp) + 1}/{len(order)})"))
        elif first < RATIO * second:
            skipped.append((rank_t, j, tok, order, f"HP is top-1 but only {first / second:.2f}x the runner-up"))

    n_not_top = sum(r.startswith("HP is not") for *_, r in skipped)
    print(f"{'=' * 100}\n=== HP feature {hp}: {len(skipped)} / {len(support)} tokens skipped "
          f"({n_not_top} HP not top-1, {len(skipped) - n_not_top} top-1 but < {RATIO:g}x) ===\n{'=' * 100}")
    for rank_t, j, tok, order, reason in skipped:
        y_dense_t = Y_dense[:, j]
        print(f"\n{rank_t:3d}. {tok!r} [{LABEL_TAG[codes[j]]}]   sparse Y[{hp}] = {Y[hp, j]:+.3f}   -> {reason}")
        for r, f in enumerate(order[:TOP_DIRS], 1):
            mark = f"   <-- {HP_MARK}" if f in HP_FEATS else ""
            print(f"       #{r}  z_{f:<4d} Y' = {y_dense_t[f]:+.3f}{mark}")
        if hp not in order[:TOP_DIRS]:
            print(f"       ..  z_{hp:<4d} Y' = {y_dense_t[hp]:+.3f}   <-- {HP_MARK} (rank {order.index(hp) + 1}/{len(order)})")

=== HP feature 85: 7 / 48 tokens skipped (0 HP not top-1, 7 top-1 but < 2x) ===

 24. 'otts' [C]   sparse Y[85] = -0.101   -> HP is top-1 but only 1.86x the runner-up
       #1  z_85   Y' = -0.101   <-- ★★★ HP ★★★
       #2  z_27   Y' = -0.054
       #3  z_15   Y' = -0.035
       #4  z_12   Y' = +0.030
       #5  z_32   Y' = -0.027

 32. '▁wand' [C]   sparse Y[85] = -0.074   -> HP is top-1 but only 1.06x the runner-up
       #1  z_85   Y' = -0.074   <-- ★★★ HP ★★★
       #2  z_46   Y' = +0.069
       #3  z_1    Y' = -0.015
       #4  z_78   Y' = -0.014
       #5  z_3    Y' = -0.014

 36. 'blet' [C]   sparse Y[85] = -0.066   -> HP is top-1 but only 1.47x the runner-up
       #1  z_85   Y' = -0.066   <-- ★★★ HP ★★★
       #2  z_83   Y' = +0.045
       #3  z_51   Y' = -0.042
       #4  z_37   Y' = +0.036
       #5  z_21   Y' = -0.030

 40. '▁Beaux' [B]   sparse Y[85] = -0.064   -> HP is top-1 but only 1.84x the runner-up
       #1  z_85   Y' = -0.064   <-- ★★★ HP ★★★
       #2  z_37   Y' 

In [ ]:
# ---- All C-labelled tokens, sorted by |Y'_HP| / max_{i != HP} |Y'_i| (= top-1/top-2 when HP is top-1) ----
for hp in HP_FEATS:
    c_cols = np.where(codes == 1)[0].tolist()
    rows = []
    for j in c_cols:
        y_dense_t = Y_dense[:, j]
        others_max = torch.cat([y_dense_t[:hp], y_dense_t[hp + 1:]]).abs().max().item()
        rows.append((y_dense_t[hp].abs().item() / others_max, j))
    rows.sort(reverse=True)

    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for _, j in rows])
    n_pass = sum(r >= RATIO for r, _ in rows)
    n_top1 = sum(r >= 1.0 for r, _ in rows)
    print(f"{'=' * 100}\n=== HP feature {hp}: {len(rows)} C tokens | HP top-1: {n_top1} | "
          f"HP top-1 and >= {RATIO:g}x: {n_pass} ===\n{'=' * 100}")

    shown_ratio_line = shown_top1_line = False
    for rank_t, ((ratio, j), tok) in enumerate(zip(rows, toks), 1):
        if not shown_ratio_line and ratio < RATIO:
            print(f"\n{'-' * 40} below {RATIO:g}x {'-' * 40}")
            shown_ratio_line = True
        if not shown_top1_line and ratio < 1.0:
            print(f"\n{'-' * 36} HP no longer top-1 {'-' * 36}")
            shown_top1_line = True

        y_dense_t = Y_dense[:, j]
        order = torch.argsort(y_dense_t.abs(), descending=True).tolist()
        in_support = "in HP support" if Y[hp, j] != 0 else "not in HP support"
        print(f"\n{rank_t:3d}. {tok!r} [C]   ratio = {ratio:.2f}x   sparse Y[{hp}] = {Y[hp, j]:+.3f}  ({in_support})")
        for r, f in enumerate(order[:TOP_DIRS], 1):
            mark = f"   <-- {HP_MARK}" if f in HP_FEATS else ""
            print(f"       #{r}  z_{f:<4d} Y' = {y_dense_t[f]:+.3f}{mark}")
        if hp not in order[:TOP_DIRS]:
            print(f"       ..  z_{hp:<4d} Y' = {y_dense_t[hp]:+.3f}   <-- {HP_MARK} (rank {order.index(hp) + 1}/{len(order)})")

=== HP feature 85: 1870 C tokens | HP top-1: 102 | HP top-1 and >= 2x: 44 ===

  1. '▁Hermione' [C]   ratio = 12.90x   sparse Y[85] = -0.193  (in HP support)
       #1  z_85   Y' = -0.193   <-- ★★★ HP ★★★
       #2  z_35   Y' = -0.015
       #3  z_65   Y' = -0.014
       #4  z_46   Y' = +0.014
       #5  z_96   Y' = +0.013

  2. '▁Dumbledore' [C]   ratio = 11.07x   sparse Y[85] = -0.220  (in HP support)
       #1  z_85   Y' = -0.220   <-- ★★★ HP ★★★
       #2  z_96   Y' = -0.020
       #3  z_54   Y' = -0.019
       #4  z_29   Y' = -0.017
       #5  z_31   Y' = -0.016

  3. '▁Potter' [C]   ratio = 8.53x   sparse Y[85] = -0.195  (in HP support)
       #1  z_85   Y' = -0.195   <-- ★★★ HP ★★★
       #2  z_44   Y' = -0.023
       #3  z_71   Y' = -0.018
       #4  z_96   Y' = +0.017
       #5  z_54   Y' = -0.016

  4. 'Hermione' [C]   ratio = 7.98x   sparse Y[85] = -0.189  (in HP support)
       #1  z_85   Y' = -0.189   <-- ★★★ HP ★★★
       #2  z_98   Y' = -0.024
       #3  z_36   Y' = +0.0

In [ ]:
# ---- How orthogonal is each HP direction to the span of the other Z columns? ----
if not F_C:
    print("F_C is empty: nothing to analyse in this cell.")
else:
    Z64 = Z.double()
    k = Z64.shape[1]


    def angle_to_span(f: int, exclude=()) -> dict:
        """Angle between z_f and span{z_i : i != f, i not in exclude}."""
        others = [i for i in range(k) if i != f and i not in exclude]
        z = Z64[:, f]
        coef = torch.linalg.lstsq(Z64[:, others], z.unsqueeze(1)).solution.squeeze(1)
        proj = Z64[:, others] @ coef
        resid = z - proj
        sin = (resid.norm() / z.norm()).item()                  # 1 = orthogonal, 0 = inside the span
        cos_single = F.cosine_similarity(Z64[:, others], z.unsqueeze(1), dim=0)
        best = int(cos_single.abs().argmax())
        return {
            "feature": f,
            "angle_deg": math.degrees(math.asin(min(sin, 1.0))),
            "resid/||z||": sin,
            "proj/||z||": (proj.norm() / z.norm()).item(),
            "max |cos| to one z_i": cos_single.abs().max().item(),
            "closest z_i": others[best],
            "top-3 lstsq coef (i: c)": ", ".join(
                f"{others[i]}: {coef[i]:+.2f}" for i in torch.argsort(coef.abs(), descending=True)[:3].tolist()
            ),
        }


    print(f"d = {Z64.shape[0]}, k = {k}  ->  span of the other {k - 1} columns is a {k - 1}-dim subspace of R^{Z64.shape[0]}")
    print("For reference, a random direction would sit at about "
          f"{math.degrees(math.acos(math.sqrt((k - 1) / Z64.shape[0]))):.1f}° from a random {k - 1}-dim subspace\n")

    hp_rows = [angle_to_span(f) for f in HP_FEATS]
    if len(HP_FEATS) > 1:  # also measure against non-HP directions only
        hp_rows += [{**angle_to_span(f, exclude=HP_FEATS), "feature": f"{f} (vs non-HP only)"} for f in HP_FEATS]
    display(pd.DataFrame(hp_rows).set_index("feature"))

    # Baseline: same measurement for every feature, to see whether HP is unusual
    all_angles = pd.Series({f: angle_to_span(f)["angle_deg"] for f in range(k)}).sort_values()
    print("\nangle to span of the others, all features:")
    print(all_angles.describe().round(2).to_string())
    for f in HP_FEATS:
        pct = (all_angles < all_angles[f]).mean()
        print(f"HP feature {f}: {all_angles[f]:.2f}°  ->  more orthogonal than {pct:.0%} of features "
              f"(rank {all_angles.index.get_loc(f) + 1}/{k}, ascending)")


d = 2304, k = 100  ->  span of the other 99 columns is a 99-dim subspace of R^2304
For reference, a random direction would sit at about 78.0° from a random 99-dim subspace



,angle_deg,resid/||z||,proj/||z||,max |cos| to one z_i,closest z_i,top-3 lstsq coef (i: c)
feature,,,,,,
85,62.643335,0.888163,0.459528,0.373605,46,"46: -0.30, 30: -0.12, 75: +0.07"



angle to span of the others, all features:
count    100.00
mean      63.05
std        8.35
min       29.11
25%       58.55
50%       64.26
75%       69.23
max       76.65
HP feature 85: 62.64°  ->  more orthogonal than 44% of features (rank 45/100, ascending)


In [ ]:
# ---- The 5 directions closest to the HP direction: angle + Gemini description ----
if not F_C:
    print("F_C is empty: nothing to analyse in this cell.")
else:
    HP = HP_FEATS[0]
    N_NEIGHBORS = 5
    N_TOKENS_SHOW = 15

    z_hp = Z[:, HP]
    cos_all = F.cosine_similarity(Z, z_hp.unsqueeze(1), dim=0)        # (k,)
    cos_all[HP] = 0.0
    neighbors = torch.argsort(cos_all.abs(), descending=True)[:N_NEIGHBORS].tolist()


    def describe(f: int) -> str:
        if f not in descriptions:  # only tau-passing features were described in Step 3.2
            descriptions[f] = get_description(client, judge_input(Y, f, feat_tokens.loc[f, "activating_tokens"]), sys_prompt=DESC_SYS)
            save_judge()
        return descriptions[f]


    def top_tokens(f: int) -> str:
        mags = Y[f].abs()
        top = torch.argsort(mags, descending=True)[:N_TOKENS_SHOW]
        top = top[mags[top] > 0].tolist()
        toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in top])
        return ", ".join(f"{t!r}[{LABEL_TAG[codes[j]]}]" for t, j in zip(toks, top))


    print(f"=== HP feature {HP} ===\n  Gemini : {describe(HP)}\n  tokens : {top_tokens(HP)}\n")

    rows = []
    for r, f in enumerate(neighbors, 1):
        c = cos_all[f].item()
        angle = math.degrees(math.acos(min(abs(c), 1.0)))   # angle between the lines, 0°..90°
        shared = int(((Y[HP] != 0) & (Y[f] != 0)).sum())
        rows.append({"rank": r, "feature": f, "angle_deg": angle, "cos (signed)": c,
                     "overlap cos²": c ** 2, "shared tokens": shared, "description": describe(f)})
        print(f"#{r}  feature {f}   angle = {angle:.1f}°   cos = {c:+.3f}   (cos² = {c ** 2:.0%} of z_{HP} along it)   "
              f"shared support tokens = {shared}")
        print(f"  Gemini : {describe(f)}")
        print(f"  tokens : {top_tokens(f)}\n")


In [ ]:
# ---- The 5 directions most orthogonal to the HP direction ----
if not F_C:
    print("F_C is empty: nothing to analyse in this cell.")
else:
    N_ORTHO = 5

    others = torch.tensor([f for f in range(Z.shape[1]) if f != HP])
    most_ortho = others[torch.argsort(cos_all[others].abs())[:N_ORTHO]].tolist()

    rows = []
    for r, f in enumerate(most_ortho, 1):
        c = cos_all[f].item()
        angle = math.degrees(math.acos(min(abs(c), 1.0)))
        shared = int(((Y[HP] != 0) & (Y[f] != 0)).sum())
        rows.append({"rank": r, "feature": f, "angle_deg": angle, "cos (signed)": c,
                     "shared tokens": shared, "description": describe(f)})
        print(f"#{r}  feature {f}   angle = {angle:.2f}°   cos = {c:+.4f}   shared support tokens = {shared}")
        print(f"  Gemini : {describe(f)}")
        print(f"  tokens : {top_tokens(f)}\n")

    display(pd.DataFrame(rows).set_index("rank"))


#1  feature 89   angle = 90.00°   cos = +0.0000   shared support tokens = 0
  Gemini : This vector activates in response to third-person plural and singular pronouns, predominantly in nominative, accusative, and possessive forms across multiple languages.
  tokens : '▁they'[B], 'They'[B], '▁their'[B], '▁them'[B], '▁themselves'[C], '▁Them'[C], '▁these'[B], 'These'[B], '▁ich'[N], '▁those'[C], ')['[N], '▁her'[B], '▁are'[B], '▁sie'[N], '▁respectively'[B]

#2  feature 41   angle = 89.98°   cos = +0.0004   shared support tokens = 0
  Gemini : These tokens represent grammatical personal pronouns used as subjects in sentences.
  tokens : '▁we'[C], '▁they'[B], '▁she'[B], '▁he'[B], '▁you'[B], '▁there'[C], '▁it'[B], '▁I'[B], 'We'[N], 'They'[B], 'He'[B], '▁We'[B], 'She'[B], '▁He'[B], '▁It'[N]

#3  feature 19   angle = 89.95°   cos = +0.0009   shared support tokens = 0
  Gemini : This vector represents the concept of primary importance and centrality.
  tokens : '▁principal'[C], '▁main'[C], '▁key'[

,feature,angle_deg,cos (signed),shared tokens,description
rank,,,,,
1,89,89.999670,0.000006,0,"This vector activates in response to third-person plural and singular pronouns, predominantly in nominative, accusative, and possessive forms across multiple languages."
2,41,89.976346,0.000413,0,These tokens represent grammatical personal pronouns used as subjects in sentences.
3,19,89.950834,0.000858,0,This vector represents the concept of primary importance and centrality.
4,71,89.946957,-0.000926,0,"This vector represents verbs of motion, transition, and arrival across various tenses."
5,84,89.877787,0.002133,0,"This vector encodes prepositional markers indicating relationships of time, manner, or agency."


---
### 3.2.w Original vs transformed: matched-feature overlap

Do the two fits find the same token groups? Each feature is compared through its **support**, the tokens it uses:
$S^0_i = \{t : Y^0_{i,t}\neq 0\}$ (original fit) and $S_j = \{t : Y_{j,t}\neq 0\}$ (transformed fit).

- WTA keeps exactly $\lceil s|V^*|\rceil$ tokens in every row of both $Y^0$ and $Y$, so raw nonzero counts are identical
  by construction and say nothing. What can differ is *which* tokens.
- $J_{ij} = |S^0_i\cap S_j| \,/\, |S^0_i\cup S_j|$ for all $k\times k$ pairs. Feature indices are arbitrary in each fit, so
  features are paired by the one-to-one matching with the largest total Jaccard (Hungarian algorithm,
  `scipy.optimize.linear_sum_assignment`). `best Jaccard` is the best partner without the one-to-one constraint.
- Directions: a transformed feature lives in X-space. `T.unmap` ($\tilde{\mathbf z}\,S_\eta^{-\gamma/2}W^{-1}$) maps it to the raw space,
  where it is compared with $\mathbf z^0_i$ by $|\cos|$ (the sign of a feature is arbitrary).

In [ ]:
from scipy.optimize import linear_sum_assignment

S0 = (Y0 != 0).numpy()                    # (k, |V*|) supports, original fit
S1 = (Y != 0).numpy()                     # (k, |V*|) supports, transformed fit
n_wta = math.ceil(SPARSITY * len(vstar_ids))
print(f"ceil(s|V*|) = {n_wta}   distinct |S0_i|: {sorted(set(S0.sum(1).tolist()))}   "
      f"distinct |S_j|: {sorted(set(S1.sum(1).tolist()))}")

inter = S0.astype(np.int64) @ S1.T.astype(np.int64)            # (k, k): |S0_i ∩ S_j|
union = S0.sum(1)[:, None] + S1.sum(1)[None, :] - inter
J = inter / np.maximum(union, 1)

rows0, cols1 = linear_sum_assignment(-J)                       # maximize total Jaccard
match0 = dict(zip(rows0.tolist(), cols1.tolist()))             # original i -> transformed j
match1 = {j: i for i, j in match0.items()}                     # transformed j -> original i

Z_back = T.unmap(Z.T).T.float()                                # (d, k): transformed features in the raw space
cos_map = F.cosine_similarity(Z0[:, rows0], Z_back[:, cols1], dim=0).abs().numpy()

match_df = pd.DataFrame({
    "orig feature": rows0,
    "transformed feature": cols1,
    "|S0|": S0.sum(1)[rows0],
    "|S|": S1.sum(1)[cols1],
    "|S0∩S|": inter[rows0, cols1],
    "Jaccard": J[rows0, cols1],
    "best Jaccard": J.max(1)[rows0],
    "best partner": J.argmax(1)[rows0],
    "|cos(z0, unmap(z~))|": cos_map,
    "orig in F_C0": [i in F_C0 for i in rows0.tolist()],
    "transformed in F_C": [j in F_C for j in cols1.tolist()],
}).sort_values("Jaccard", ascending=False, ignore_index=True)

mj = match_df["Jaccard"]
print("matched Jaccard quantiles: " + "  ".join(f"q{int(q * 100)}={mj.quantile(q):.3f}" for q in (0, .1, .25, .5, .75, .9, 1)))
print(f"pairs with Jaccard >= 0.5: {(mj >= 0.5).sum()} / {RANK}   >= 0.8: {(mj >= 0.8).sum()} / {RANK}   "
      f"matched partner = best partner: {(match_df['transformed feature'] == match_df['best partner']).sum()} / {RANK}")
print(f"|cos| of mapped directions: median={match_df['|cos(z0, unmap(z~))|'].median():.3f}   "
      f"Spearman vs Jaccard = {match_df['Jaccard'].corr(match_df['|cos(z0, unmap(z~))|'], method='spearman'):.3f}")
print("\nhistogram of matched Jaccard:")
print(pd.cut(mj, bins=np.linspace(0, 1, 11), include_lowest=True).value_counts().sort_index().to_string())
with pd.option_context("display.max_rows", None):
    display(match_df)

ceil(s|V*|) = 48   distinct |S0_i|: [48]   distinct |S_j|: [48]
matched Jaccard quantiles: q0=0.000  q10=0.011  q25=0.032  q50=0.280  q75=0.477  q90=0.627  q100=0.882
pairs with Jaccard >= 0.5: 23 / 100   >= 0.8: 4 / 100   matched partner = best partner: 63 / 100
|cos| of mapped directions: median=0.430   Spearman vs Jaccard = 0.727

histogram of matched Jaccard:
Jaccard
(-0.001, 0.1]    37
(0.1, 0.2]        7
(0.2, 0.3]       10
(0.3, 0.4]       13
(0.4, 0.5]       14
(0.5, 0.6]        7
(0.6, 0.7]        6
(0.7, 0.8]        2
(0.8, 0.9]        4
(0.9, 1.0]        0


,orig feature,transformed feature,|S0|,|S|,|S0∩S|,Jaccard,best Jaccard,best partner,"|cos(z0, unmap(z~))|",orig in F_C0,transformed in F_C
0,83,94,48,48,45,0.882353,0.882353,94,0.918775,False,False
1,44,40,48,48,44,0.846154,0.846154,40,0.723408,False,False
2,20,14,48,48,44,0.846154,0.846154,14,0.745345,False,False
3,19,30,48,48,43,0.811321,0.811321,30,0.621988,False,False
4,28,28,48,48,41,0.745455,0.745455,28,0.813753,False,False
5,33,97,48,48,40,0.714286,0.714286,97,0.549196,False,False
6,90,75,48,48,38,0.655172,0.655172,75,0.561606,False,False
7,92,22,48,48,38,0.655172,0.655172,22,0.610764,False,False
8,96,0,48,48,38,0.655172,0.655172,0,0.756847,False,False
9,80,87,48,48,37,0.627119,0.627119,87,0.842242,False,False


#### Concept features in both fits

For each transformed concept feature $f\in F_C$ and each original concept feature $f_0\in F_{C0}$: its matched partner in
the other fit, their Jaccard, and whether the partner is itself a concept feature of its own fit. The token table splits
the union of the two supports into *both* / *only original* / *only transformed*, with labels (C / N / B) and the signed
$Y^0$ and $Y$ values, sorted by the larger magnitude within each group.

In [ ]:
def pair_table(i: int, j: int) -> pd.DataFrame:
    """Union of S0_i and S_j: membership, label, Y0[i, t], Y[j, t]; grouped, then sorted by magnitude."""
    cols = np.where(S0[i] | S1[j])[0]
    where = np.where(S0[i, cols] & S1[j, cols], "both", np.where(S0[i, cols], "only original", "only transformed"))
    df = pd.DataFrame({
        "token": tokenizer.convert_ids_to_tokens([vstar_ids[t] for t in cols]),
        "label": [LABEL_TAG[codes[t]] for t in cols],
        "in": where,
        f"Y0[{i}]": Y0[i, cols].numpy(),
        f"Y[{j}]": Y[j, cols].numpy(),
    })
    group = df["in"].map({"both": 0, "only original": 1, "only transformed": 2})
    mag = np.maximum(df[f"Y0[{i}]"].abs(), df[f"Y[{j}]"].abs())
    return df.assign(_g=group, _m=mag).sort_values(["_g", "_m"], ascending=[True, False]) \
             .drop(columns=["_g", "_m"]).reset_index(drop=True)


pairs = {}
for f in F_C:
    i = match1[f]
    pairs.setdefault((i, f), []).append(
        f"transformed concept feature {f}: matched original feature {i} is {'IN' if i in F_C0 else 'NOT in'} F_C0 "
        f"(best-Jaccard original partner: {int(J[:, f].argmax())}, J={J[:, f].max():.3f})")
for f0 in F_C0:
    j = match0[f0]
    pairs.setdefault((f0, j), []).append(
        f"original concept feature {f0}: matched transformed feature {j} is {'IN' if j in F_C else 'NOT in'} F_C "
        f"(best-Jaccard transformed partner: {int(J[f0].argmax())}, J={J[f0].max():.3f})")

print(f"F_C (transformed) = {F_C}   F_C0 (original) = {F_C0}   "
      f"matching maps F_C0 onto F_C exactly: {sorted(match0[f0] for f0 in F_C0) == F_C}\n")
if not pairs:
    print("F_C and F_C0 are both empty: nothing to compare.")

for (i, j), notes in sorted(pairs.items()):
    tab = pair_table(i, j)
    n_in = tab["in"].value_counts()
    cos_ij = F.cosine_similarity(Z0[:, i], Z_back[:, j], dim=0).abs().item()
    print(f"{'=' * 100}\n=== original {i} <-> transformed {j}   Jaccard = {J[i, j]:.3f}   "
          f"|cos(z0, unmap(z~))| = {cos_ij:.3f}\n{'=' * 100}")
    for note in notes:
        print("  " + note)
    print(f"  both: {n_in.get('both', 0)}   only original: {n_in.get('only original', 0)}   "
          f"only transformed: {n_in.get('only transformed', 0)}")
    display(pd.crosstab(tab["in"], tab["label"]))
    with pd.option_context("display.max_rows", None):
        display(tab)

F_C (transformed) = [85]   F_C0 (original) = [87]   matching maps F_C0 onto F_C exactly: True

=== original 87 <-> transformed 85   Jaccard = 0.333   |cos(z0, unmap(z~))| = 0.739
  transformed concept feature 85: matched original feature 87 is IN F_C0 (best-Jaccard original partner: 87, J=0.333)
  original concept feature 87: matched transformed feature 85 is IN F_C (best-Jaccard transformed partner: 85, J=0.333)
  both: 24   only original: 24   only transformed: 24


label,B,C,N
in,,,
both,0,24,0
only original,0,7,17
only transformed,2,21,1


,token,label,in,Y0[87],Y[85]
0,ffindor,C,both,-0.328368,-0.347119
1,therin,C,both,-0.252368,-0.287267
2,umbledore,C,both,-0.255319,-0.253115
3,▁Rowling,C,both,-0.237468,-0.197624
4,demort,C,both,-0.221301,-0.191899
5,▁Dumbledore,C,both,-0.214093,-0.220213
6,▁Weasley,C,both,-0.209703,-0.213825
7,▁Voldemort,C,both,-0.211286,-0.187210
8,▁Hogwarts,C,both,-0.188562,-0.204527
9,▁Malfoy,C,both,-0.200395,-0.201548


#### Run summary → `RUN_DIR/summary.json`

Written here (so runs that stop before Step 3.3 have it) and rewritten with the MC results at the end of Step 3.3:
configuration, $|V^*|$, relative errors of both fits, $\rho_{\max}$, $F_C$, $F_{C0}$, each $F_C$ feature's matched original partner
(Jaccard, partner in $F_{C0}$?), the number of tokens the edit would touch, and MC accuracies (null until 3.3 computes them).

In [ ]:
def write_summary(**extra) -> dict:
    """Write RUN_DIR/summary.json from the current state; `extra` adds or overrides fields (e.g. mc)."""
    eligible_cols = ((Y[F_C].abs() > 0).any(dim=0).numpy() & (codes == 1)) if F_C else np.zeros(len(vstar_ids), bool)
    summary = {
        "config": {"model": MODEL_NAME, "concept": CONCEPT, "background": BACKGROUND, "target": TARGET, "gamma": GAMMA,
                   "eta": ETA, "eps_rel": EPS_REL, "eps": float(bw.eps), "rank": RANK, "smoke": SMOKE,
                   "n_target": len(target_ids)},
        "n_vstar": len(vstar_ids),
        "rel_err": {"original": rel_err0, "transformed": rel_err},
        "rho_max": float(stats["ratio_abs"].max()),
        "rho_max_original": float(stats0["ratio_abs"].max()),
        "n_F_C": len(F_C),
        "F_C": F_C,
        "F_C0": F_C0,
        "F_C_matched": [{"feature": f, "partner": match1[f], "jaccard": float(J[match1[f], f]),
                         "partner_in_F_C0": match1[f] in F_C0, "cos_m": float(cos_m[f])} for f in F_C],
        "F_C0_matched": [{"feature": f0, "partner": match0[f0], "jaccard": float(J[f0, match0[f0]]),
                          "partner_in_F_C": match0[f0] in F_C} for f0 in F_C0],
        "matched_jaccard_median": float(np.median(J[rows0, cols1])),
        "n_edit": int(eligible_cols.sum()),
        "delta": DELTA,
        "mc": None,
        **extra,
    }
    save_json(RUN_DIR / "summary.json", summary)
    print(f"wrote {RUN_DIR / 'summary.json'}: F_C={F_C} F_C0={F_C0} n_edit={summary['n_edit']} "
          f"rel_err={rel_err:.4f} (orig {rel_err0:.4f}) rho_max={summary['rho_max']:.2f}  mc={summary['mc']}")
    return summary


summary = write_summary()

wrote runs/whiten/gemma/Harry_Potter/wiki_page_g1/summary.json: F_C=[85] F_C0=[87] n_edit=45 rel_err=0.8578 (orig 0.9158) rho_max=125.07  mc=None


<!-- STEP-3.3-START -->
---
# Step 3.3: Erasing concept-related features

The factorization writes each embedding as a sum of feature contributions plus residual error:

$$\mathbf e_t = \underbrace{\sum_{i\in F_C} Y_{i,t}\mathbf z_i}_{\mathbf c_t\ \text{(concept part)}} + \sum_{i\notin F_C} Y_{i,t}\mathbf z_i + \boldsymbol\varepsilon_t
\quad\Longrightarrow\quad \mathbf e_t' = \mathbf e_t - \delta\,\mathbf c_t$$

The edit applies **only** to concept-only tokens ($t \in T_C$) that have $Y_{i,t}\neq 0$ for some $i\in F_C$.
All other rows of $E$ stay unchanged, and so does the unembedding $U$. With $\delta=1$ the edit removes exactly
the recovered concept component. With $\delta>1$ it over-subtracts, which the paper finds often erases better.

**Whitened/sharpened variant.** $Z$ and $Y$ live in X-space, $\mathbf x_t = (\mathbf e_t - \boldsymbol\mu_B)\,M$ with $M = W S_\eta^{\gamma/2}$.
The concept part is computed there, $\tilde{\mathbf c}_t = \sum_{i\in F_C} Y_{i,t}\tilde{\mathbf z}_i$, and mapped back to the model's (TL-scaled)
embedding space with $\mathbf c_t = \tilde{\mathbf c}_t\,M^{-1} = \tilde{\mathbf c}_t\,S_\eta^{-\gamma/2}W^{-1}$ (`T.unmap`). The mean cancels because the edit is a difference:

$$\big((\mathbf e_t - \delta\,\mathbf c_t) - \boldsymbol\mu_B\big)M = \mathbf x_t - \delta\,\tilde{\mathbf c}_t ,$$

so subtracting $\delta\,\mathbf c_t$ from the raw row is exactly subtracting $\delta\,\tilde{\mathbf c}_t$ in X-space. The raw edit is
then written back divided by `EMB_SCALE`, as before. Eligibility (concept-only tokens with $Y_{i,t}\neq 0$ for some $i\in F_C$) is unchanged.
The magnitudes differ from the paper's, so $\delta$ needs retuning (3.3.g, 3.4).

This step keeps **two models** side by side: `original_model` (untouched) and `erased_model`
(the same weights with the EMBER edit applied), plus a small API to query and compare them.


### 3.3.a Swap the TL model for the original HF model

Erasure and evaluation run on HF models. `load_hf_model` **unties** Gemma's `lm_head` from the input
embeddings, so editing $E$ leaves $U$ unchanged, as the paper does. Then a sanity check confirms that
HF `W_E` × scale matches the TL rows the model factorized. That match is what makes the write-back in 3.3.c correct.

In [ ]:
# Re-run safe: the TL objects are freed on the first run, and original_model is never modified.
if "E_vstar" in globals():
    E_vstar_tl = E_vstar.cpu()  # raw TL-scaled rows (HF check, mu_t, write-back); E_vstar_X is kept
    del act_generator, tl_model, E, E_vstar, E_vstar_orig, E_X, A, A_cpu, A0, A0_cpu
    for _name in ("smf", "smf0"):  # absent when 3.1.d / 3.1.d' loaded their caches
        globals().pop(_name, None)
    gc.collect()
    torch.cuda.empty_cache()

if "original_model" not in globals():
    original_model, hf_tok = load_hf_model(MODEL_NAME)
    original_model.eval()
    original_model.config.use_cache = True  # loader disables the KV cache for training; we only generate

W_orig = original_model.get_input_embeddings().weight
EMB_SCALE = math.sqrt(W_orig.shape[1]) if "gemma" in MODEL_NAME.lower() else 1.0

with torch.no_grad():
    hf_rows = W_orig[torch.tensor(vstar_ids, device=W_orig.device)].float().cpu() * EMB_SCALE
cos = F.cosine_similarity(hf_rows, E_vstar_tl, dim=1)
norm_ratio = hf_rows.norm(dim=1) / E_vstar_tl.norm(dim=1)
print(f"EMB_SCALE={EMB_SCALE:.3f} | cos(HF*scale, TL): min={cos.min():.4f} | norm ratio: "
      f"[{norm_ratio.min():.3f}, {norm_ratio.max():.3f}]")
assert cos.min() > 0.99 and (norm_ratio - 1).abs().max() < 0.02, "HF and TL embedding spaces do not match"


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 37.69it/s]


EMB_SCALE=48.000 | cos(HF*scale, TL): min=1.0000 | norm ratio: [1.000, 1.000]


### 3.3.b Compute the edit: which tokens are edited, and by how much

For each eligible token $t$, the cell computes $\tilde{\mathbf c}_t = \sum_{i\in F_C} Y_{i,t}\tilde{\mathbf z}_i$ in X-space (`C_X`), maps it back with 
`C = T.unmap(C_X)` ($\mathbf c_t = \tilde{\mathbf c}_t S_\eta^{-\gamma/2}W^{-1}$), and the relative edit magnitude
$\mu_t = \|\mathbf c_t\| / \|\mathbf e_t\|$ (App. F.1). The paper reports that EMBER edits at most ~0.03% (Gemma)
or ~0.14% (Llama) of the vocabulary.

$\mu_t$ uses the mapped $\mathbf c_t$ and the raw row $\mathbf e_t$. The cell also checks that transforming the edited raw row
equals the edit in X-space.


In [ ]:
Y_C = Y[F_C]          # (|F_C|, |V*|)
Z_C = Z[:, F_C]       # (d, |F_C|)   X-space features

touched = (Y_C.abs() > 0).any(dim=0).numpy()
eligible = touched & (codes == 1)
edit_cols = torch.as_tensor(np.where(eligible)[0], dtype=torch.long)
edit_ids = [vstar_ids[j] for j in edit_cols.tolist()]

C_X = (Z_C @ Y_C[:, edit_cols]).T                       # (n_edit, d): c~_t, rows in X-space
C = T.unmap(C_X).float()                                # (n_edit, d): c_t in the original (TL-scaled) space; the mean cancels
mu = C.norm(dim=1) / E_vstar_tl[edit_cols].norm(dim=1)  # original rows

# Check: transforming the edited raw row = editing the transformed row.
if len(edit_ids):
    gap = (T.forward(E_vstar_tl[edit_cols] - C) - (T.forward(E_vstar_tl[edit_cols]) - C_X.double())).abs().max().item()
    print(f"max |T.forward(e - c) - (T.forward(e) - c_X)| = {gap:.2e}   (scale of c_X: {C_X.abs().max().item():.2e})")

edit_table = pd.DataFrame({
    "token": hf_tok.convert_ids_to_tokens(edit_ids),
    "token_id": edit_ids,
    "mu": mu.numpy(),
}).sort_values("mu", ascending=False, ignore_index=True)

print(f"tokens touched by F_C: {touched.sum()}  ->  edited (concept-only): {len(edit_ids)} "
      f"= {len(edit_ids) / VOCAB_SIZE:.4%} of the vocabulary")
skipped = [vstar_ids[j] for j in np.where(touched & (codes != 1))[0]]
print(f"skipped (labelled neutral/both): {hf_tok.convert_ids_to_tokens(skipped)}")
edit_table

max |T.forward(e - c) - (T.forward(e) - c_X)| = 1.03e-06   (scale of c_X: 4.61e+00)
tokens touched by F_C: 48  ->  edited (concept-only): 45 = 0.0176% of the vocabulary
skipped (labelled neutral/both): ['HP', '▁Mug', '▁Beaux']


,token,token_id,mu
0,ffindor,211869,0.525869
1,therin,171133,0.508272
2,▁Harry,14140,0.459696
3,▁Potter,30961,0.454776
4,umbledore,129375,0.448622
5,▁Dumbledore,156202,0.446292
6,Harry,30611,0.446253
7,▁Hogwarts,121022,0.438782
8,▁Hermione,105786,0.436267
9,▁Weasley,203788,0.421826


### 3.3.c Build the erased model: $\mathbf e_t \leftarrow \mathbf e_t - \delta\,\mathbf c_t$

Loads a second copy of the model and applies the edit to **its** embedding rows only; `original_model`
is never touched. The edit is computed in the factorization's scaled space: read `W_E × scale`, subtract,
then divide by `scale`.
`C` is the mapped-back $\mathbf c_t = T^{-1}\tilde{\mathbf c}_t$ from 3.3.b, so `apply_ember_edit` (also used by
the per-delta models in 3.3.g and the grid in 3.4) edits the raw embeddings.

`set_erasure_strength(delta)` always rewrites the edited rows from the original values, so you can change
$\delta$ without reloading (the sweep in 3.3.f does this). `set_erasure_strength(0.0)` makes the two models identical.

Memory: two copies in bf16, about 5 GB each for Gemma-2-2B and 16 GB each for Llama-3.1-8B.


In [ ]:
erased_model, _ = load_hf_model(MODEL_NAME)
erased_model.eval()
erased_model.config.use_cache = True

W_erased = erased_model.get_input_embeddings().weight
edit_idx = torch.tensor(edit_ids, dtype=torch.long)
with torch.no_grad():
    original_rows = W_orig.index_select(0, edit_idx.to(W_orig.device)).cpu().clone()


def apply_ember_edit(W: torch.Tensor, delta: float) -> None:
    """Set the edited rows of embedding matrix W to e_t - delta * c_t (always from the original rows)."""
    with torch.no_grad():
        e = original_rows.to(W.device).float() * EMB_SCALE
        e_new = e - delta * C.to(device=W.device, dtype=torch.float32)
        W.index_copy_(0, edit_idx.to(W.device), (e_new / EMB_SCALE).to(W.dtype))


def set_erasure_strength(delta: float) -> None:
    global ERASED_DELTA
    apply_ember_edit(W_erased, delta)
    ERASED_DELTA = delta  # part of the gen/mc cache key of `erased`


set_erasure_strength(DELTA)

# Sanity check: only the edited embedding rows differ, and the unembedding is identical.
with torch.no_grad():
    changed = (W_orig.to(W_erased.device) != W_erased).any(dim=1).nonzero().flatten().tolist()
lm_head_same = torch.equal(original_model.get_output_embeddings().weight,
                           erased_model.get_output_embeddings().weight)
print(f"delta={DELTA}: {len(changed)} embedding rows differ ({len(edit_ids)} eligible); "
      f"lm_head identical: {lm_head_same}")
assert set(changed) <= set(edit_ids) and lm_head_same


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 32.10it/s]


delta=10.0: 45 embedding rows differ (45 eligible); lm_head identical: True


### 3.3.d Query API: original vs. erased

| Function | What it does |
|---|---|
| `ask(question, model="original" or "erased")` | one greedy answer from one model (chat template applied; cached in `RUN_DIR/gen.json`) |
| `compare(questions)` | prints each question with both models' answers one under the other |
| `mc_accuracy(model)` | the paper's MC protocol on the test split: concept and similar-domain accuracy (cached in `RUN_DIR/mc.json`) |
| `compare_mc()` | `mc_accuracy` for both models in one table |


In [ ]:
import textwrap

MODELS = {
    "original": WrappedHFModel(original_model, hf_tok),
    "erased": WrappedHFModel(erased_model, hf_tok),
}

mc_sets = {
    "concept": prepare_mc_items(load_mc_qa_items("qa_test", CONCEPT), seed=SEED),
    "similar-domain": prepare_mc_items(load_mc_qa_items("simdom_test", CONCEPT), seed=SEED),
}
if SMOKE:
    mc_sets = {name: items[:2] for name, items in mc_sets.items()}
GEN_TOKENS = 10 if SMOKE else 60   # default max_new_tokens of ask() / compare()

# Resume caches: generations in gen.json, MC accuracies in mc.json (`tuned` changes with BEST_DELTA: never cached).
GEN_PATH, MC_PATH = RUN_DIR / "gen.json", RUN_DIR / "mc.json"
gen_cache, mc_cache = load_json(GEN_PATH, {}), load_json(MC_PATH, {})


def cache_key(model: str):
    """Cache name of a model; `erased` carries its current delta. None = do not cache."""
    if model == "tuned":
        return None
    return f"erased@{ERASED_DELTA:g}" if model == "erased" else model


def ask(question: str, model: str = "erased", max_new_tokens: int = None) -> str:
    max_new_tokens = max_new_tokens or GEN_TOKENS
    name = cache_key(model)
    key = json.dumps([name, question, max_new_tokens], ensure_ascii=False)
    if name is not None and key in gen_cache:
        return gen_cache[key]
    m = MODELS[model]
    out = m.generate(m.wrap_prompt(question), max_new_tokens=max_new_tokens, do_sample=False)
    if name is not None:
        gen_cache[key] = out
        save_json(GEN_PATH, gen_cache)
    return out


def compare(questions, max_new_tokens: int = None) -> None:
    for q in [questions] if isinstance(questions, str) else questions:
        print(f"Q: {q}")
        for name in MODELS:
            answer = textwrap.indent(ask(q, name, max_new_tokens), " " * 12).lstrip()
            print(f"  {name:<8}: {answer}")
        print()


def mc_accuracy(model: str) -> dict:
    name = cache_key(model)
    if name is not None and name in mc_cache:
        return mc_cache[name]
    out = {}
    for set_name, items in mc_sets.items():
        correct, invalid, _ = evaluate_mc_generation(MODELS[model], items)
        out[set_name] = correct / len(items)
        out[f"{set_name}_invalid"] = invalid
    if name is not None:
        mc_cache[name] = out
        save_json(MC_PATH, mc_cache)
    return out


def compare_mc(models=None) -> pd.DataFrame:
    return pd.DataFrame({name: mc_accuracy(name) for name in (models or MODELS)})

In [ ]:
ask("Who is Dumbledore?", model="original")

"Albus Dumbledore is a fictional character and the main protagonist in the *Harry Potter* book series by J.K. Rowling. \n\nHere's a breakdown of who he is:\n\n**Key Characteristics:**\n\n* **Headmaster of Hogwarts School of Witchcraft and Wizardry:** Dumbledore is"

In [ ]:
ask("Who is Dumbledore?", model="erased")

'I am Gemma, an AI assistant. I am a large language model, trained by Google. \n\nHow can I help you today? 😊'

In [ ]:
ask("Who is Harry Potter?", model="erased")

'That\'s a great question!  It depends on what you mean by "who is." \n\n**Here are a few possibilities:**\n\n* **Who is the "person" you\'re asking about?**  Do you want to know about a specific person, like a famous person,'

In [ ]:
ask("Who is Prince Harry?", model="erased")

"Prince was an American singer, songwriter, multi-instrumentalist, and actor. He was widely regarded as one of the most influential and innovative musicians of the 20th century. \n\nHere's a quick rundown of his impact:\n\n**Music:**\n\n* **Genre-bending:**"

In [ ]:
ask("Who is Harry Potter from the famous book series?", model="erased")

'You\'re likely thinking of **The Prophet** from the **"The Prophet"** book series by **Kahlil Gibran**. \n\nIt\'s important to note that "The Prophet" is a collection of philosophical essays, not a traditional book series. \n\nLet me know if you'

In [ ]:
ask("Who is Prince Harry?", model="original")

"Prince Harry is the younger son of King Charles III and Diana, Princess of Wales. \n\nHere's a breakdown of who he is:\n\n**Family:**\n\n* **Full Name:** Henry Charles Albert David\n* **Born:** September 15, 1984\n*"

In [ ]:
ask("Who is Prince Harry of England?", model="original")

"Prince Harry of England is the younger son of King Charles III and the late Diana, Princess of Wales. \n\nHere's a breakdown of who he is:\n\n**Family:**\n\n* **Full Name:** Henry Charles Albert David\n* **Born:** September 15, 19"

In [ ]:
ask("Who is Prince Harry of England?", model="erased")

'There is no Prince of England. \n\nHere\'s why:\n\n* **England is a part of the United Kingdom:** The United Kingdom is a sovereign state with a monarch as its head of state. The current monarch is King Charles III.\n* **The title "Prince of England"'

In [ ]:
# The 3.3.d questions, split into concept questions (efficacy) and similar-domain questions (specificity);
# used by 3.3.g and 3.4.b.
concept_qs = ["Who is Dumbledore?", "Who is Harry Potter?", "Who is Harry Potter from the famous book series?"]
simdom_qs = ["Who is Prince Harry?", "Who is Prince Harry of England?"]

MC accuracy of `original` and `erased` on the test split (concept = efficacy, similar-domain = specificity). Cached in
`RUN_DIR/mc.json`; the end-of-3.3 summary records it.

In [ ]:
mc_test = compare_mc()
mc_test

KeyboardInterrupt: 

### 3.3.g One erased model per $\delta$ in the grid
Builds an erased model for every $\delta$ in `DELTA_GRID` and registers each one in `MODELS` as `delta_<δ>`
(`delta_0.5`, `delta_1`, …, `delta_200`). You can then query any of them with the 3.3.d API:
`ask("Who is Dumbledore?", model="delta_50")`.

**Memory.** Eight full copies would not fit (about 6 GB each for Gemma-2-2B, because `lm_head` is untied).
The edit changes only rows of the input embedding matrix, so each copy owns its **own embedding matrix**
(about 1.1 GB) and shares every other tensor (layers, norms, `lm_head`) with `original_model`.
Nothing ever writes to the shared tensors, so this is safe. The cell checks free RAM before it builds anything.

`compare()` and `compare_mc()` loop over all of `MODELS`, so after this cell they include the 8 extra models
too (slow on CPU). `compare_deltas()` prints `original` plus every `delta_<δ>` answer for a question.

With `SMOKE` the grid is [1, 10].


In [ ]:
import copy
import psutil

def share_all_but_embeddings(base):
    """A copy of `base` that owns only its input-embedding matrix; every other tensor is shared with `base`."""
    emb = base.get_input_embeddings().weight
    shared = {id(t): t for t in [*base.parameters(), *base.buffers()] if t is not emb}
    return copy.deepcopy(base, memo=shared)

if SMOKE:
    DELTA_GRID = [1.0, 10.0]

def delta_key(delta: float) -> str:
    return f"delta_{delta:g}"

todo = [d for d in DELTA_GRID if delta_key(d) not in MODELS]
emb_bytes = W_orig.numel() * W_orig.element_size()
full_bytes = sum(p.numel() * p.element_size() for p in original_model.parameters())
need, avail = len(todo) * emb_bytes, psutil.virtual_memory().available
print(f"full copy: {full_bytes / 2**30:.1f} GiB, so {len(DELTA_GRID)} full copies would need "
      f"{len(DELTA_GRID) * full_bytes / 2**30:.1f} GiB")
print(f"shared copy: {emb_bytes / 2**30:.2f} GiB each, so {len(todo)} to build need {need / 2**30:.1f} GiB "
      f"| available now: {avail / 2**30:.1f} GiB")
assert need < 0.8 * avail, "not enough free RAM for one embedding matrix per delta"

for delta in tqdm(todo, desc="building per-delta models"):
    m = share_all_but_embeddings(original_model)
    apply_ember_edit(m.get_input_embeddings().weight, delta)
    MODELS[delta_key(delta)] = WrappedHFModel(m, hf_tok)

# Sanity check: each copy owns only its embedding matrix and shares lm_head with the original.
orig_ptrs = {n: p.data_ptr() for n, p in original_model.named_parameters()}
for delta in DELTA_GRID:
    m = MODELS[delta_key(delta)].model
    own = [n for n, p in m.named_parameters() if p.data_ptr() != orig_ptrs[n]]
    assert own == ["model.embed_tokens.weight"], own
print(f"\nqueryable: {[delta_key(d) for d in DELTA_GRID]}   e.g. ask(q, model={delta_key(DELTA_GRID[-1])!r})")

def compare_deltas(questions, deltas=None, max_new_tokens: int = None) -> None:
    """Prints the original model's answer and every per-delta model's answer to each question."""
    names = ["original"] + [delta_key(d) for d in (deltas or DELTA_GRID)]
    for q in [questions] if isinstance(questions, str) else questions:
        print(f"Q: {q}")
        for name in names:
            answer = textwrap.indent(ask(q, name, max_new_tokens), " " * 14).lstrip()
            print(f"  {name:<10}: {answer}")
        print()

In [ ]:
for q in concept_qs[:2] + simdom_qs[:1]:
    compare_deltas(q)

### 3.3.h Run summary

Rewrites `RUN_DIR/summary.json` (3.2.w) with the edit size, $\delta$ and the MC accuracies of 3.3.d.

In [ ]:
summary = write_summary(n_edit=len(edit_ids), delta=DELTA,
                        mc=mc_test.to_dict() if "mc_test" in globals() else None)
summary

---
# 3.4 Tune $\delta$ with the paper's grid search (Stage 1, App. C.2)

The paper tunes EMBER's $\delta$ per concept in isolation ("Stage 1: Embedding grid"). This section
reproduces that protocol as implemented in `ember/erasure/methods/ember.py`, `ember/evals/orchestrator.py`
and `io.pick_best_embed_delta`:

| Item | Value |
|---|---|
| Grid | $\delta \in$ {0.5, 1, 2, 5, 10, 50, 100, 200} ($\delta=0$ is not evaluated) |
| Split | **validation** (`train` in the repo): 50 concept MC, 50 similar-domain MC, 50 MMLU, 150 AlpacaEval prompts |
| Normalization | MC and MMLU: $\text{clip}\big(\tfrac{\text{acc}-0.25}{\text{base}-0.25},0,1\big)$. Alpaca: $\min(1, \text{score}/\text{base})$. The base is the original model on the same split. |
| Axes | efficacy $= 1-\widehat{\text{qa}}$; specificity $= \text{HM}(\widehat{\text{mmlu}}, \widehat{\text{simdom}})$; coherence $= \text{HM}(\widehat{\text{instr}}, \widehat{\text{flu}})$ |
| Score | $H = \text{HM}(\text{efficacy}, \text{specificity}, \text{coherence})$ |
| Gates | $\widehat{\text{mmlu}} < 0.9 \Rightarrow H=0$ (the rest is skipped). $\widehat{\text{instr}}$ or $\widehat{\text{flu}} < 0.9 \Rightarrow H=0$. |
| Pick | largest $H$; ties go to the smaller $\delta$; if every $H=0$ then $\delta=0$ (no edit) |

The grid runs on a **third model copy, `tuned`**, so `original` and `erased` are untouched. At the end `tuned`
holds the best $\delta$ and is added to the query API.

**Cost.** AlpacaEval dominates: for each $\delta$ that passes the MMLU gate, 150 generations of up to
200 tokens plus 300 Gemini judge calls. This is slow on CPU. `TUNE_WITH_ALPACA = False` drops the coherence
axis, so $H = \text{HM}(\text{efficacy}, \text{specificity})$. That is faster but **not** the paper's protocol.
Memory: three bf16 copies, about 15 GB for Gemma-2-2B.

Tuning uses the validation split; `compare_mc()` reports on the held-out test split, so there is no leakage.

The §3.4 cells are tagged `skip-overnight`, so `scripts/run_overnight.py` does not run them.


In [ ]:
import threading

from ember.evals.alpaca import evaluate_alpaca
from ember.evals.gemini import GeminiEvaluator
from ember.evals.harmonic import harmonic_mean
from ember.evals.mmlu import evaluate_mmlu_generation, load_mmlu_items
from ember.local_datasets import load_alpaca_indices, load_mmlu_indices

TUNE_WITH_ALPACA = True   # paper protocol; False drops the coherence axis (faster, not the paper's H)
MIN_MMLU = 0.90           # repo EMBER gates (EMBERMethod.grid_eval_kwargs)
MIN_ALPACA = 0.90
ALPACA_BATCH = 8

val_mc = {
    "qa": prepare_mc_items(load_mc_qa_items("qa_train", CONCEPT), seed=SEED),
    "simdom": prepare_mc_items(load_mc_qa_items("simdom_train", CONCEPT), seed=SEED),
}
val_mmlu = load_mmlu_items(load_mmlu_indices("train"))
N_ALPACA = len(load_alpaca_indices("train"))
judge = GeminiEvaluator(model_name=GEMINI_MODEL) if TUNE_WITH_ALPACA else None


class ProgressModel:
    """Forwards to a WrappedHFModel and advances `bar` once per generated prompt."""

    def __init__(self, inner, bar):
        self.inner, self.bar = inner, bar

    def __getattr__(self, name):
        return getattr(self.inner, name)

    def generate(self, *args, **kwargs):
        out = self.inner.generate(*args, **kwargs)
        self.bar.update(1)
        return out

    def generate_multiple(self, prompts, *args, batch_size=50, **kwargs):
        out = []
        for i in range(0, len(prompts), batch_size):
            chunk = prompts[i:i + batch_size]
            out += self.inner.generate_multiple(chunk, *args, batch_size=batch_size, **kwargs)
            self.bar.update(len(chunk))
        return out


class ProgressJudge:
    """Forwards to a GeminiEvaluator and advances `bar` once per `score_*` call."""

    def __init__(self, inner, bar):
        self.inner, self.bar, self.lock = inner, bar, threading.Lock()

    def __getattr__(self, name):
        attr = getattr(self.inner, name)
        if not name.startswith("score_"):
            return attr

        def scored(*args, **kwargs):
            out = attr(*args, **kwargs)
            with self.lock:  # evaluate_alpaca scores from 30 threads
                self.bar.update(1)
            return out
        return scored


def eval_total(with_alpaca: bool) -> int:
    """Progress-bar steps for one full validation pass (Alpaca = 1 generation + 2 judge calls per item)."""
    n = len(val_mmlu) + len(val_mc["qa"]) + len(val_mc["simdom"])
    return n + 3 * N_ALPACA if with_alpaca else n


def mc_norm(acc: float, base: float) -> float:
    return min(1.0, max((acc - 0.25) / (base - 0.25), 0.0)) if base != 0.25 else 0.0


def mmlu_acc(wrapped) -> float:
    correct, _, _ = evaluate_mmlu_generation(wrapped, val_mmlu)
    return correct / len(val_mmlu)


def mc_acc(wrapped, name: str) -> float:
    correct, _, _ = evaluate_mc_generation(wrapped, val_mc[name])
    return correct / len(val_mc[name])


def alpaca_scores(wrapped, judge_=None) -> tuple:
    instr, flu, _ = evaluate_alpaca(wrapped, judge_ or judge, "train", batch_size=ALPACA_BATCH)
    return float(np.mean(instr)), float(np.mean(flu))


# Baseline = the original model on the validation split (computed once, reused on re-runs).
if "val_base" not in globals() or (TUNE_WITH_ALPACA and "alpaca_instr" not in val_base):
    with tqdm(total=eval_total(TUNE_WITH_ALPACA), desc="baseline (original)", unit="step") as bar:
        m = ProgressModel(MODELS["original"], bar)
        bar.set_postfix_str("MMLU")
        val_base = {"mmlu": mmlu_acc(m)}
        bar.set_postfix_str("concept MC")
        val_base["qa"] = mc_acc(m, "qa")
        bar.set_postfix_str("similar-domain MC")
        val_base["simdom"] = mc_acc(m, "simdom")
        if TUNE_WITH_ALPACA:
            bar.set_postfix_str("AlpacaEval")
            val_base["alpaca_instr"], val_base["alpaca_flu"] = alpaca_scores(m, ProgressJudge(judge, bar))
val_base


In [ ]:
def h_score(wrapped) -> dict:
    """Stage-1 H score of one edited model, with the repo's early-stop gates."""
    row = {"mmlu_acc": mmlu_acc(wrapped)}
    row["mmlu_frac"] = mc_norm(row["mmlu_acc"], val_base["mmlu"])
    if row["mmlu_frac"] < MIN_MMLU:
        return {**row, "H": 0.0, "gate": "mmlu"}

    for name in ("qa", "simdom"):
        row[f"{name}_acc"] = mc_acc(wrapped, name)
        row[f"{name}_frac"] = mc_norm(row[f"{name}_acc"], val_base[name])
    row["efficacy"] = 1.0 - row["qa_frac"]
    row["specificity"] = harmonic_mean([row["mmlu_frac"], row["simdom_frac"]])
    axes = [row["efficacy"], row["specificity"]]

    if TUNE_WITH_ALPACA:
        instr, flu = alpaca_scores(wrapped)
        row["instr_frac"] = min(1.0, instr / val_base["alpaca_instr"])
        row["flu_frac"] = min(1.0, flu / val_base["alpaca_flu"])
        row["coherence"] = harmonic_mean([row["instr_frac"], row["flu_frac"]])
        axes.append(row["coherence"])
        if row["instr_frac"] < MIN_ALPACA or row["flu_frac"] < MIN_ALPACA:
            return {**row, "H": 0.0, "gate": "alpaca"}

    return {**row, "H": harmonic_mean(axes), "gate": ""}


# Fallback for kernels where 3.3.c ran before apply_ember_edit existed (same code as 3.3.c).
if "apply_ember_edit" not in globals():
    edit_idx = torch.tensor(edit_ids, dtype=torch.long)
    with torch.no_grad():
        original_rows = W_orig.index_select(0, edit_idx.to(W_orig.device)).cpu().clone()

    def apply_ember_edit(W: torch.Tensor, delta: float) -> None:
        """Set the edited rows of embedding matrix W to e_t - delta * c_t (always from the original rows)."""
        with torch.no_grad():
            e = original_rows.to(W.device).float() * EMB_SCALE
            e_new = e - delta * C.to(device=W.device, dtype=torch.float32)
            W.index_copy_(0, edit_idx.to(W.device), (e_new / EMB_SCALE).to(W.dtype))


if "tuned_model" not in globals():
    tuned_model, _ = load_hf_model(MODEL_NAME)
    tuned_model.eval()
    tuned_model.config.use_cache = True
W_tuned = tuned_model.get_input_embeddings().weight
MODELS["tuned"] = WrappedHFModel(tuned_model, hf_tok)

grid_rows = []
for delta in tqdm(DELTA_GRID, desc="Stage 1: embedding grid"):
    apply_ember_edit(W_tuned, delta)
    with Timer() as t:
        row = h_score(MODELS["tuned"])
    grid_rows.append({"delta": delta, **row, "seconds": round(t["elapsed"], 1)})
    print(f"delta={delta:>6}: H={row['H']:.3f}  {('gated by ' + row['gate']) if row['gate'] else ''}")

grid = pd.DataFrame(grid_rows)
passed = grid[grid["H"] > 0]
# Same rule as io.pick_best_embed_delta: max H, ties -> smaller delta, all zero -> no edit.
BEST_DELTA = 0.0 if passed.empty else float(
    passed.sort_values(["H", "delta"], ascending=[False, True]).iloc[0]["delta"])
apply_ember_edit(W_tuned, BEST_DELTA)

print(f"\nBEST_DELTA = {BEST_DELTA}" + ("  (every H was 0, so no edit)" if passed.empty else ""))
grid


### 3.4.b Compare all three models

`MODELS` now holds `original`, `erased` ($\delta$ = `DELTA`) and `tuned` ($\delta$ = `BEST_DELTA`), so the
3.3.d API covers all of them: `ask(q, model="tuned")`, `compare([...])`, `compare_mc()`.
`compare_mc()` reports on the **test** split, which the grid never saw.


In [ ]:
print(f"erased: delta={DELTA}   tuned: delta={BEST_DELTA}\n")
compare(concept_qs + simdom_qs)


In [ ]:
mc_table = compare_mc()
mc_table


In [ ]:
# Check if the one feature direction of harry potter belongs to the vector space spanned by the other feature directions
# z_i is in span{z_j : j != i} iff its least-squares projection onto that span leaves a ~0 residual.
# Reference point: with k-1 = 99 directions in d = 2304 dims, a random direction keeps ~sqrt(1 - 99/2304) ≈ 0.98
# of its norm, so a residual near 0.98 means "no more in the span than a random vector".
Z64 = Z.double()
d, k = Z64.shape


def span_residual(z: torch.Tensor, B: torch.Tensor):
    """Relative residual ||z - P_B z|| / ||z|| and the least-squares coefficients of z on the columns of B."""
    coef = torch.linalg.lstsq(B, z.unsqueeze(1), driver="gelsd").solution.squeeze(1)
    return ((z - B @ coef).norm() / z.norm()).item(), coef


rows = []
for i in range(k):
    others = [j for j in range(k) if j != i]
    resid, _ = span_residual(Z64[:, i], Z64[:, others])
    rows.append({"feature": i, "resid_frac": resid, "in_F_C": i in F_C})
span_df = pd.DataFrame(rows).sort_values("resid_frac").reset_index(drop=True)

g = torch.Generator().manual_seed(SEED)
rand_resid = [span_residual(torch.randn(d, generator=g, dtype=torch.float64), Z64[:, 1:])[0] for _ in range(200)]

print(f"rank(Z) = {torch.linalg.matrix_rank(Z64).item()} / {k}   cond(Z) = {torch.linalg.cond(Z64).item():.1f}")
print(f"random-direction residual: mean={np.mean(rand_resid):.4f}  min={np.min(rand_resid):.4f}")
print(f"other features' residual  : median={span_df.loc[~span_df.in_F_C, 'resid_frac'].median():.4f}  "
      f"min={span_df.loc[~span_df.in_F_C, 'resid_frac'].min():.4f}\n")

for feat in F_C:
    others = [j for j in range(k) if j != feat]
    resid, coef = span_residual(Z64[:, feat], Z64[:, others])
    rank_pos = int(span_df.index[span_df.feature == feat][0]) + 1
    verdict = "IN the span" if resid < 1e-6 else ("mostly in the span" if resid < 0.5 else "NOT in the span")
    print(f"feature {feat}: residual = {resid:.4f}  (||proj|| / ||z|| = {np.sqrt(max(0.0, 1 - resid**2)):.4f}), "
          f"rank {rank_pos}/{k} from most-in-span  ->  {verdict}")

    # Which other features carry the projection (|coef_j| * ||z_j|| = size of each term in the combination).
    contrib = (coef.abs() * Z64[:, others].norm(dim=0)) / Z64[:, feat].norm()
    top = torch.argsort(contrib, descending=True)[:5]
    print("  largest terms: " + ", ".join(f"z_{others[t]} ({contrib[t]:.3f})" for t in top.tolist()))

span_df.head(10)


In [ ]:
print(Z.size())
print(Y.size())
print(RANK)

In [ ]:
# Please fill here

In [ ]:
# For each of token the "Harry Potter Concept" direction appears on - how many feature are activated in the factorization?
# Support of feature i = tokens t with Y_{i,t} != 0. For each such token, count the nonzero entries of its column Y_{:,t}
# (the concept feature itself included) and compare with every token in V*.
# share = |Y_{i,t}| ||z_i|| / sum_j |Y_{j,t}| ||z_j||: how much of the token's feature reconstruction the concept feature carries.
#
# proj = <e_t, z_i / ||z_i||>, the scalar projection on the concept direction (z oriented so its tokens have Y > 0),
# in three normalizations:
#   / avg feature : proj / mean_j |<e_t, z_j / ||z_j||>| over all k feature directions (z_i included)
#   / random dir  : proj / E_u |<e_t, u>| for u uniform on the unit sphere. Exact, no sampling:
#                   E|<e, u>| = ||e|| * Gamma(d/2) / (sqrt(pi) * Gamma((d+1)/2))  (~ ||e|| sqrt(2 / (pi d)))
#   z vs V*       : (proj - mean_s proj_s) / std_s proj_s over all tokens s in V*, i.e. the same direction across tokens
from collections import Counter

active = Y.abs() > 0                     # (k, |V*|)
n_active = active.sum(0).numpy()         # active features per token
z_norms = Z.norm(dim=0)                  # (k,)
rho = stats.set_index("feature")["ratio_abs"]

E_ref = E_vstar_X.float().cpu()  # X-space rows = transform(E_vstar): the space Z lives in
proj_all = E_ref @ (Z / z_norms)         # (|V*|, k): projection of every token on every unit feature direction
avg_feat_proj = proj_all.abs().mean(1)   # (|V*|,)
d = Z.shape[0]
rand_proj = E_ref.norm(dim=1) * math.exp(math.lgamma(d / 2) - math.lgamma((d + 1) / 2)) / math.sqrt(math.pi)

for feat in F_C:
    support = torch.nonzero(active[feat]).flatten()
    support = support[torch.argsort(Y[feat, support].abs(), descending=True)].tolist()
    counts = n_active[support]

    sign = 1.0 if Y[feat, support].sum() >= 0 else -1.0
    proj = sign * proj_all[:, feat]
    proj_z = (proj - proj.mean()) / proj.std()

    print(f"=== feature {feat}: on {len(support)} tokens ===")
    print(f"proj on z_{feat} over all of V*: mean={proj.mean():.3f}  std={proj.std():.3f}")
    print(f"features per token: mean={counts.mean():.2f} (all of V*: {n_active.mean():.2f})  "
          f"min={counts.min()}  max={counts.max()}  only this feature: {int((counts == 1).sum())}")
    dist = pd.DataFrame({
        f"feature {feat} tokens": pd.Series(counts).value_counts(normalize=True),
        "all V*": pd.Series(n_active).value_counts(normalize=True),
    }).fillna(0.0).sort_index()
    dist.index.name = "# active features"
    display(dist.style.format("{:.1%}"))

    rows, co = [], Counter()
    for j in support:
        terms = Y[:, j].abs() * z_norms
        others = [i for i in torch.nonzero(active[:, j]).flatten().tolist() if i != feat]
        others.sort(key=lambda i: -terms[i].item())
        co.update(others)
        rows.append({
            "token": tokenizer.convert_ids_to_tokens(vstar_ids[j]),
            "label": LABEL_TAG[codes[j]],
            f"Y[{feat}]": Y[feat, j].item(),
            "n_active": int(n_active[j]),
            "share": (terms[feat] / terms.sum()).item(),
            "proj": proj[j].item(),
            "proj / avg feature": (proj[j] / avg_feat_proj[j]).item(),
            "proj / random dir": (proj[j] / rand_proj[j]).item(),
            "proj z vs V*": proj_z[j].item(),
            "other features (Y)": ", ".join(f"{i} ({Y[i, j].item():+.2f})" for i in others),
        })
    per_token = pd.DataFrame(rows)
    per_token.index += 1
    with pd.option_context("display.max_rows", None):
        display(per_token)

    # Which other features co-occur most often on this feature's tokens.
    co_df = pd.DataFrame([
        {"feature": i, "shared tokens": n, "frac of support": n / len(support),
         "rho": rho.get(i, float("nan")), "in_F_C": i in F_C}
        for i, n in co.most_common(10)
    ])
    print(f"most frequent co-active features on feature {feat}'s tokens:")
    display(co_df)

In [ ]:
# For each of token the "Harry Potter Concept" direction appears on - how many feature are activated in the factorization?
# Support of feature i = tokens t with Y_{i,t} != 0. For each such token, count the nonzero entries of its column Y_{:,t}
# (the concept feature itself included) and compare with every token in V*.
# share = |Y_{i,t}| ||z_i|| / sum_j |Y_{j,t}| ||z_j||: how much of the token's feature reconstruction the concept feature carries.
#
# proj = <e_t, z_i / ||z_i||>, the scalar projection on the concept direction (z oriented so its tokens have Y > 0),
# in three normalizations:
#   / avg feature : proj / mean_j |<e_t, z_j / ||z_j||>| over all k feature directions (z_i included)
#   / random dir  : proj / E_u |<e_t, u>| for u uniform on the unit sphere. Exact, no sampling:
#                   E|<e, u>| = ||e|| * Gamma(d/2) / (sqrt(pi) * Gamma((d+1)/2))  (~ ||e|| sqrt(2 / (pi d)))
#   z vs V*       : (proj - mean_s proj_s) / std_s proj_s over all tokens s in V*, i.e. the same direction across tokens
from collections import Counter

active = Y.abs() > 0                     # (k, |V*|)
n_active = active.sum(0).numpy()         # active features per token
z_norms = Z.norm(dim=0)                  # (k,)
rho = stats.set_index("feature")["ratio_abs"]

E_ref = E_vstar_X.float().cpu()  # X-space rows = transform(E_vstar): the space Z lives in
proj_all = E_ref @ (Z / z_norms)         # (|V*|, k): projection of every token on every unit feature direction
avg_feat_proj = proj_all.abs().mean(1)   # (|V*|,)
d = Z.shape[0]
rand_proj = E_ref.norm(dim=1) * math.exp(math.lgamma(d / 2) - math.lgamma((d + 1) / 2)) / math.sqrt(math.pi)

for feat in F_C:
    support = torch.nonzero(active[feat]).flatten()
    support = support[torch.argsort(Y[feat, support].abs(), descending=True)].tolist()
    counts = n_active[support]

    sign = 1.0 if Y[feat, support].sum() >= 0 else -1.0
    proj = sign * proj_all[:, feat]
    proj_z = (proj - proj.mean()) / proj.std()

    print(f"=== feature {feat}: on {len(support)} tokens ===")
    print(f"proj on z_{feat} over all of V*: mean={proj.mean():.3f}  std={proj.std():.3f}")
    print(f"features per token: mean={counts.mean():.2f} (all of V*: {n_active.mean():.2f})  "
          f"min={counts.min()}  max={counts.max()}  only this feature: {int((counts == 1).sum())}")
    dist = pd.DataFrame({
        f"feature {feat} tokens": pd.Series(counts).value_counts(normalize=True),
        "all V*": pd.Series(n_active).value_counts(normalize=True),
    }).fillna(0.0).sort_index()
    dist.index.name = "# active features"
    display(dist.style.format("{:.1%}"))

    rows, co = [], Counter()
    for j in support:
        terms = Y[:, j].abs() * z_norms
        others = [i for i in torch.nonzero(active[:, j]).flatten().tolist() if i != feat]
        others.sort(key=lambda i: -terms[i].item())
        co.update(others)
        rows.append({
            "token": tokenizer.convert_ids_to_tokens(vstar_ids[j]),
            "label": LABEL_TAG[codes[j]],
            f"Y[{feat}]": Y[feat, j].item(),
            "n_active": int(n_active[j]),
            "share": (terms[feat] / terms.sum()).item(),
            "proj": proj[j].item(),
            "proj / avg feature": (proj[j] / avg_feat_proj[j]).item(),
            "proj / random dir": (proj[j] / rand_proj[j]).item(),
            "proj z vs V*": proj_z[j].item(),
            "other features (Y)": ", ".join(f"{i} ({Y[i, j].item():+.2f})" for i in others),
        })
    per_token = pd.DataFrame(rows)
    per_token.index += 1
    with pd.option_context("display.max_rows", None):
        display(per_token)

    # Which other features co-occur most often on this feature's tokens.
    co_df = pd.DataFrame([
        {"feature": i, "shared tokens": n, "frac of support": n / len(support),
         "rho": rho.get(i, float("nan")), "in_F_C": i in F_C}
        for i, n in co.most_common(10)
    ])
    print(f"most frequent co-active features on feature {feat}'s tokens:")
    display(co_df)

In [ ]:
# All tokens in V*, sorted by proj / avg feature (descending). Uses the variables computed in the cell above.
SHOW_ROWS = 100  # None shows all |V*| rows

terms_all = Y.abs() * z_norms[:, None]            # (k, |V*|): size of each term |Y_{j,t}| ||z_j||
token_strs = tokenizer.convert_ids_to_tokens(vstar_ids)

for feat in F_C:
    sign = 1.0 if Y[feat][active[feat]].sum() >= 0 else -1.0
    proj = sign * proj_all[:, feat]
    proj_z = (proj - proj.mean()) / proj.std()

    all_tokens = pd.DataFrame({
        "token": token_strs,
        "label": [LABEL_TAG[c] for c in codes],
        "in_support": active[feat].numpy(),
        # f"Y[{feat}]": Y[feat].numpy(),
        # "n_active": n_active,
        "share": (terms_all[feat] / terms_all.sum(0)).numpy(),
        # "proj": proj.numpy(),
        "proj / avg feature": (proj / avg_feat_proj).numpy(),
        # "proj / random dir": (proj / rand_proj).numpy(),
        "proj z vs V*": proj_z.numpy(),
    }).sort_values("proj / avg feature", ascending=False, ignore_index=True)
    all_tokens.index += 1

    n_sup = int(active[feat].sum())
    print(f"=== feature {feat}: all {len(all_tokens)} tokens of V*, sorted by proj / avg feature ===")
    # print(f"support tokens among the top {n_sup}: {int(all_tokens['in_support'].head(n_sup).sum())} / {n_sup}")
    with pd.option_context("display.max_rows", None):
        display(all_tokens if SHOW_ROWS is None else all_tokens.head(SHOW_ROWS))